In [ ]:
import sys, re, json, gzip, bz2, time, zipfile, tarfile, random, shutil, subprocess, traceback, urllib.request
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

WORK = Path("/content/r2_sources")          # temporary Colab disk, NOT Google Drive
WORK.mkdir(exist_ok=True)
URLS = {
    "fews": "https://nlp.cs.washington.edu/fews/fews.zip",
    "wikt_it": "https://kaikki.org/itwiktionary/raw-wiktextract-data.jsonl.gz",
    "wikt_es": "https://kaikki.org/eswiktionary/raw-wiktextract-data.jsonl.gz",
    "wikt_nl": "https://kaikki.org/nlwiktionary/raw-wiktextract-data.jsonl.gz",
    "wikt_sv": "https://dumps.wikimedia.org/svwiktionary/latest/svwiktionary-latest-pages-articles.xml.bz2",
    "axolotl": "https://github.com/ltgoslo/axolotl24_shared_task.git",
}
XLWSD_GDRIVE_ID = "19YTL-Uq95hjiFZfgwEpXRgcYGCR_PQY0"
SPLITS = PROJECT / "outputs/runs/step07_frozen_splits/20261105T220410Z_0c2b5b52/splits.json"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
EN_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a/EN_wic_dev.jsonl"
SEED = 2027
norm = lambda w: (w or "").strip().casefold()

def download(url, dest, min_bytes=1_000_000):
    if dest.exists() and dest.stat().st_size >= min_bytes:
        return dest
    part = dest.with_name(dest.name + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "semeval2027-class-project/1.0 (research use)"})
    with urllib.request.urlopen(request, timeout=180) as response, open(part, "wb") as handle:
        shutil.copyfileobj(response, handle, 1 << 20)
    assert part.stat().st_size >= min_bytes, f"Download too small ({part.stat().st_size} bytes): {url}"
    part.replace(dest)
    return dest

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def slot(n_seen, k, rng):
    """Uniform random sample of k items from a stream: where to store item number n_seen, or None."""
    if n_seen <= k:
        return n_seen - 1
    j = rng.randrange(n_seen)
    return j if j < k else None

def show(sentence, start, end):
    return (sentence[:start] + "<t>" + sentence[start:end] + "</t>" + sentence[end:])[:260]

# ---------------------------------------------------------------------------
section("0. WORDS THAT MUST NEVER ENTER TRAINING (our judges)")
SV_DEV_WORDS = ("dumpa", "exportera", "fru", "fröken", "förort", "herre", "kvinna", "motiv", "skär", "suga")
SV_STEMS = ("dump", "export", "fru", "fröken", "förort", "herr", "kvinn", "motiv", "skär", "sug")
word_lists = json.loads(SPLITS.read_text())["word_lists"]
EN_HELD = {norm(w) for w in word_lists.get("WiC_EN_validation", [])} | {norm(r["word"]) for r in read_jsonl(EN_EVAL)}
XL_HELD = {code.lower(): {norm(r["word"]) for r in read_jsonl(XL_EVAL / f"{code}_xlwsd_test.jsonl")} for code in ("IT", "ES", "NL")}
FI_TRAIN_WORDS = {norm(w) for w in word_lists.get("AXOLOTL_FI_train", [])}
print(f"Swedish dev words (never trained on)      : {len(SV_DEV_WORDS)}; any Swedish Wiktionary title containing one of")
print(f"                                            {SV_STEMS} is excluded (this also drops compounds).")
print(f"English evaluation words held out         : {len(EN_HELD):,}")
for code, held in XL_HELD.items():
    print(f"{code.upper()} XL-WSD test lemmas held out           : {len(held):,}")
print(f"Finnish words in the frozen training split: {len(FI_TRAIN_WORDS):,}")

summary, t_all = {}, time.perf_counter()
def run(name, function):
    section(name)
    t0 = time.perf_counter()
    try:
        summary[name] = dict(function(), status="OK")
    except Exception as error:                       # one failed source must not stop the others
        traceback.print_exc(limit=4)
        summary[name] = dict(status="FAILED", error=repr(error)[:300], usable=0, words=0)
        print(f"\nFAILED: {name}. The other sources continue. Paste this error with the output.")
    print(f"[{time.perf_counter() - t0:.0f}s]", flush=True)

# ---------------------------------------------------------------------------
def check_fews():
    archive = zipfile.ZipFile(download(URLS["fews"], WORK / "fews.zip", 10_000_000))
    names = archive.namelist()
    senses_name = next(n for n in names if n.endswith("senses.txt"))
    train_name = next(n for n in names if n.endswith("train/train.txt"))
    senses, block = {}, {}
    for line in archive.read(senses_name).decode("utf-8").splitlines():
        line = line.strip()
        if not line:
            if "sense_id" in block:
                senses[block["sense_id"]] = block
            block = {}
            continue
        key, _, value = line.partition(":\t")
        block[key.rstrip(":")] = value.strip()
    if "sense_id" in block:
        senses[block["sense_id"]] = block
    family = defaultdict(list)                     # senses of the same word and part of speech
    for sense_id, s in senses.items():
        if s.get("gloss"):
            family[sense_id.rsplit(".", 1)[0]].append(sense_id)
    c, words, samples, rng = Counter(), set(), [], random.Random(SEED)
    for line in archive.read(train_name).decode("utf-8").splitlines():
        parts = line.strip().split("\t")
        if len(parts) != 2:
            c["skip_bad_line"] += 1
            continue
        sentence, label = parts
        c["train_examples"] += 1
        if sentence.count("<WSD>") != 1 or sentence.count("</WSD>") != 1:
            c["skip_not_exactly_one_marked_word"] += 1
            continue
        if label not in senses or not senses[label].get("gloss"):
            c["skip_no_definition"] += 1
            continue
        siblings = [x for x in family[label.rsplit(".", 1)[0]] if x != label and senses[x]["gloss"] != senses[label]["gloss"]]
        if not siblings:
            c["skip_no_other_sense"] += 1
            continue
        plain = sentence.replace("<WSD>", "").replace("</WSD>", "")
        if not 20 <= len(plain) <= 500:
            c["skip_length"] += 1
            continue
        word = norm(senses[label].get("word"))
        if word in EN_HELD:
            c["skip_held_out_word"] += 1
            continue
        c["usable"] += 1
        words.add(word)
        i = slot(c["usable"], 4, rng)
        if i is not None:
            samples[i:i + 1] = [(senses[label]["word"], sentence.replace("<WSD>", "<t>").replace("</WSD>", "</t>")[:260],
                                 senses[label]["gloss"][:150], senses[rng.choice(siblings)]["gloss"][:150])]
    print(f"Senses with a definition: {sum(len(v) for v in family.values()):,} | training examples: {c['train_examples']:,}")
    print(f"Usable (one marked word, definition, at least one other sense, not a held-out word): {c['usable']:,} over {len(words):,} words")
    print("Skipped:", {k: v for k, v in c.items() if k.startswith("skip")})
    for word, sentence, yes, no in samples:
        print(f"\n  word: {word}\n  sentence: {sentence}\n  YES def : {yes}\n  NO def  : {no}")
    return dict(language="EN", usable=c["usable"], words=len(words), license="CC BY-NC 4.0 (FEWS repository licence)",
                labels="Wiktionary quotations, sense chosen by Wiktionary editors", counts=dict(c))

# ---------------------------------------------------------------------------
TOKEN = re.compile(r"\w+")
def locate(text, offsets, forms):
    if isinstance(offsets, list) and len(offsets) == 1 and isinstance(offsets[0], (list, tuple)) and len(offsets[0]) == 2:
        a, b = offsets[0]
        if isinstance(a, int) and isinstance(b, int) and 0 <= a < b <= len(text):
            return (a, b), "bold"
    hits = [(m.start(), m.end()) for m in TOKEN.finditer(text) if m.group(0).casefold() in forms]
    return (hits[0], "form") if len(hits) == 1 else (None, None)

def check_wiktionary(code, language):
    path = download(URLS[f"wikt_{code}"], WORK / f"{code}wiktionary.jsonl.gz", 5_000_000)
    held = XL_HELD[code]
    c, top_keys, sense_keys, example_keys = Counter(), Counter(), Counter(), Counter()
    words, samples, rng = set(), [], random.Random(SEED)
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            try:
                entry = json.loads(line)
            except ValueError:
                c["bad_json_lines"] += 1
                continue
            if not isinstance(entry, dict) or entry.get("lang_code") != code:
                continue
            c["entries"] += 1
            probe = c["entries"] <= 5000
            if probe:
                top_keys.update(entry.keys())
            word = entry.get("word") or ""
            senses = []
            for s in entry.get("senses") or []:
                if probe:
                    sense_keys.update(s.keys())
                glosses = [g.strip() for g in (s.get("glosses") or []) if isinstance(g, str) and g.strip()]
                if not glosses:
                    continue
                if s.get("form_of") or s.get("alt_of") or "form-of" in (s.get("tags") or []):
                    c["skip_form_of_senses"] += 1
                    continue
                senses.append((glosses[-1], [e for e in (s.get("examples") or []) if isinstance(e, dict)]))
            c["senses_with_definition"] += len(senses)
            n_examples = sum(len(e) for _, e in senses)
            c["examples_total"] += n_examples
            if len({g for g, _ in senses}) < 2:
                c["skip_examples_word_has_one_sense"] += n_examples
                continue
            forms = {norm(word)} | {norm(f.get("form")) for f in (entry.get("forms") or []) if isinstance(f, dict) and f.get("form")}
            forms = {f for f in forms if len(f) >= 2 and " " not in f}
            for gloss, examples in senses:
                for e in examples:
                    if probe:
                        example_keys.update(e.keys())
                    text = (e.get("text") or "").strip()
                    if not 20 <= len(text) <= 400:
                        c["skip_length"] += 1
                        continue
                    span, how = locate(text, e.get("bold_text_offsets"), forms)
                    if span is None:
                        c["skip_target_word_not_found_once"] += 1
                        continue
                    if norm(word) in held:
                        c["skip_held_out_word"] += 1
                        continue
                    c["usable"] += 1
                    c[f"target_found_by_{how}"] += 1
                    words.add(norm(word))
                    i = slot(c["usable"], 4, rng)
                    if i is not None:
                        other = rng.choice(sorted({g for g, _ in senses} - {gloss}))
                        samples[i:i + 1] = [(word, show(text, *span), gloss[:150], other[:150])]
    assert c["entries"] > 0, f"No entries with lang_code == '{code}' found: the file format differs from what was expected."
    print(f"{language} entries: {c['entries']:,} | senses with a definition: {c['senses_with_definition']:,} | example sentences: {c['examples_total']:,}")
    print(f"Usable (word has 2+ senses, target word found exactly once, not a held-out word): {c['usable']:,} over {len(words):,} words")
    print(f"  target found by bold markup: {c['target_found_by_bold']:,} | by exact word-form match: {c['target_found_by_form']:,}")
    print("Skipped:", {k: v for k, v in c.items() if k.startswith("skip")})
    print("Fields seen: entry", sorted(top_keys)[:14], "| sense", sorted(sense_keys)[:10], "| example", sorted(example_keys)[:8])
    for word, sentence, yes, no in samples:
        print(f"\n  word: {word}\n  sentence: {sentence}\n  YES def : {yes}\n  NO def  : {no}")
    return dict(language=code.upper(), usable=c["usable"], words=len(words), license="CC BY-SA (Wiktionary), extracted by kaikki.org",
                labels="example listed under its own sense; NO = another sense of the same entry (automatic)", counts=dict(c))

# ---------------------------------------------------------------------------
BOLD = "\x01"
def clean_wikitext(text):
    text = re.sub(r"<!--.*?-->|<ref[^>]*/>|<ref[^>]*>.*?</ref>", "", text, flags=re.S)
    previous = None
    while previous != text:                        # remove {{templates}}, innermost first
        previous, text = text, re.sub(r"\{\{[^{}]*\}\}", "", text)
    text = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", text)
    return text

def clean_definition(text):
    return re.sub(r"\s+", " ", clean_wikitext(text).replace("'''", "").replace("''", "")).strip(" .;:,")

def clean_example(text):
    """Returns (plain sentence, start, end) when exactly one '''bold''' target is present, else None."""
    text = clean_wikitext(text).replace("'''", BOLD).replace("''", "")
    text = re.sub(r"\s+", " ", text).strip()
    if text.count(BOLD) != 2:
        return None
    start = text.index(BOLD)
    end = text.index(BOLD, start + 1) - 1
    plain = text.replace(BOLD, "")
    return (plain, start, end) if 0 <= start < end <= len(plain) else None

def swedish_blocks(wikitext):
    """Yields (part of speech, [(raw definition, [raw example lines])]) for the Swedish section of a page."""
    match = re.search(r"^==\s*Svenska\s*==\s*$", wikitext, flags=re.M)
    if not match:
        return
    rest = wikitext[match.end():]
    nxt = re.search(r"^==[^=].*?==\s*$", rest, flags=re.M)
    body = rest[:nxt.start()] if nxt else rest
    pos, definitions = None, []
    for line in body.splitlines():
        heading = re.match(r"^===\s*([^=].*?)\s*===\s*$", line)
        if heading:
            if pos and definitions:
                yield pos, definitions
            pos, definitions = heading.group(1), []
        elif line.startswith("#") and not line.startswith(("#:", "#*", "##")):
            definitions.append((line[1:], []))
        elif line.startswith("#:") and definitions:
            body_line = line[2:].strip()
            if body_line.startswith("''"):          # italic line = example sentence; '#:{{...}}' lines are metadata
                definitions[-1][1].append(body_line)
    if pos and definitions:
        yield pos, definitions

def check_swedish():
    path = download(URLS["wikt_sv"], WORK / "svwiktionary.xml.bz2", 5_000_000)
    c, words, samples, raw_shown, rng = Counter(), set(), [], [], random.Random(SEED)
    with bz2.open(path, "rb") as handle:
        context = ET.iterparse(handle, events=("start", "end"))
        _, root = next(context)
        for event, element in context:
            if event != "end" or element.tag.rsplit("}", 1)[-1] != "page":
                continue
            title = ns = text = None
            for child in element.iter():
                tag = child.tag.rsplit("}", 1)[-1]
                if tag == "title":
                    title = child.text
                elif tag == "ns":
                    ns = child.text
                elif tag == "text":
                    text = child.text
            root.clear()
            c["pages"] += 1
            if ns != "0" or not title or not text or "Svenska" not in text:
                continue
            blocks = list(swedish_blocks(text))
            if not blocks:
                continue
            c["swedish_entries"] += 1
            if title in ("bil", "springa") and len(raw_shown) < 2:
                start = text.find("==Svenska==")
                raw_shown.append((title, text[max(start, 0):max(start, 0) + 700]))
            excluded = any(stem in title.casefold() for stem in SV_STEMS)
            for pos, definitions in blocks:
                cleaned = [(clean_definition(d), examples) for d, examples in definitions]
                cleaned = [(d, examples) for d, examples in cleaned if len(d) >= 3]
                c["definitions"] += len(cleaned)
                n_examples = sum(len(e) for _, e in cleaned)
                c["example_lines"] += n_examples
                if len({d for d, _ in cleaned}) < 2:
                    c["skip_examples_word_has_one_sense"] += n_examples
                    continue
                for definition, examples in cleaned:
                    for raw in examples:
                        parsed = clean_example(raw)
                        if parsed is None:
                            c["skip_no_single_bold_target"] += 1
                            continue
                        plain, a, b = parsed
                        if not 15 <= len(plain) <= 400:
                            c["skip_length"] += 1
                            continue
                        if excluded:
                            c["skip_dev_word_or_compound"] += 1
                            continue
                        c["usable"] += 1
                        words.add(norm(title))
                        i = slot(c["usable"], 5, rng)
                        if i is not None:
                            other = rng.choice(sorted({d for d, _ in cleaned} - {definition}))
                            samples[i:i + 1] = [(title, pos, show(plain, a, b), definition[:150], other[:150])]
    assert c["swedish_entries"] > 0, "No '==Svenska==' sections found: the dump format differs from what was expected."
    print(f"Pages read: {c['pages']:,} | Swedish entries: {c['swedish_entries']:,} | definitions: {c['definitions']:,} | example lines: {c['example_lines']:,}")
    print(f"Usable (2+ senses, one bold target word, not a dev word or compound): {c['usable']:,} over {len(words):,} words")
    print("Skipped:", {k: v for k, v in c.items() if k.startswith("skip")})
    leaked = sorted(w for w in words if any(stem in w for stem in SV_STEMS))
    print(f"Dev-word leak check (must be empty): {leaked} -> {'PASS' if not leaked else 'FAIL'}")
    assert not leaked
    for title, raw in raw_shown:
        print(f"\n  RAW PAGE TEXT for '{title}' (to confirm the page layout):\n" + "\n".join("    " + l for l in raw.splitlines()[:16]))
    for title, pos, sentence, yes, no in samples:
        print(f"\n  word: {title} ({pos})\n  sentence: {sentence}\n  YES def : {yes}\n  NO def  : {no}")
    return dict(language="SV", usable=c["usable"], words=len(words), license="CC BY-SA (Swedish Wiktionary dump)",
                labels="example listed under its own sense; NO = another sense of the same entry (automatic)", counts=dict(c))

# ---------------------------------------------------------------------------
def check_xlwsd():
    archive_path = WORK / "xl-wsd-data.zip"
    if not archive_path.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
        result = gdown.download(id=XLWSD_GDRIVE_ID, output=str(archive_path), quiet=True)
        assert result and archive_path.is_file(), "XL-WSD download failed."
    if zipfile.is_zipfile(archive_path):
        z = zipfile.ZipFile(archive_path)
        names, read = set(z.namelist()), z.read
    else:
        t = tarfile.open(archive_path)
        names, read = set(t.getnames()), (lambda n: t.extractfile(n).read())

    def sentences(xml_bytes):
        yield from ET.fromstring(xml_bytes).iter("sentence")
    def gold_keys(path):
        out = {}
        for line in read(path).decode("utf-8").splitlines():
            parts = line.split()
            if len(parts) >= 2:
                out[parts[0]] = parts[1:]
        return out
    def definitions(lang):
        base = f"xl-wsd/training_datasets/wngt_glosses_{lang}/wngt_glosses_{lang}"
        keys, out = gold_keys(base + ".gold.key.txt"), {}
        for s in sentences(read(base + ".data.xml")):
            tokens = list(s)
            colon = next((i for i, t in enumerate(tokens) if (t.text or "").strip() == ":"), None)
            if colon is None:
                continue
            codes = {code for t in tokens[:colon] if t.tag == "instance" and t.attrib.get("id") in keys for code in keys[t.attrib["id"]]}
            text = " ".join((t.text or "") for t in tokens[colon + 1:]).strip()
            if len(codes) == 1 and text:
                out.setdefault(codes.pop(), text)
        return out
    def inventory(lang):
        out = {}
        for line in read(f"xl-wsd/inventories/inventory.{lang}.txt").decode("utf-8").splitlines():
            parts = line.split("\t")
            if len(parts) >= 2:
                out[parts[0]] = parts[1:]
        return out
    def count(folder, lang, defs, inv, held):
        data = next((n for n in names if n.startswith(f"xl-wsd/{folder}/") and n.endswith(".data.xml")), None)
        if data is None:
            return None
        gold, c, lemmas = gold_keys(data[:-len(".data.xml")] + ".gold.key.txt"), Counter(), set()
        for s in sentences(read(data)):
            for t in s:
                if t.tag != "instance":
                    continue
                c["instances"] += 1
                senses = gold.get(t.attrib.get("id"))
                right = next((g for g in senses or [] if g in defs), None)
                if right is None:
                    c["skip_no_definition"] += 1
                    continue
                key = f"{t.attrib.get('lemma')}#{t.attrib.get('pos')}"
                if not any(x not in senses and x in defs and defs[x] != defs[right] for x in inv.get(key, [])):
                    c["skip_no_other_sense"] += 1
                    continue
                lemma = norm((t.attrib.get("lemma") or "").replace("_", " "))
                if lemma in held:
                    c["skip_held_out_lemma"] += 1
                    continue
                c["usable"] += 1
                lemmas.add(lemma)
        return c, lemmas

    total, total_words, detail = 0, 0, {}
    print(f"{'lang':<5}{'source':<34}{'instances':>11}{'usable':>9}{'words':>8}   skipped")
    for lang in ("it", "es", "nl"):
        defs, inv = definitions(lang), inventory(lang)
        for label, folder in (("dev split (human-labelled)", f"evaluation_datasets/dev-{lang}"),
                              ("translated SemCor (silver)", f"training_datasets/semcor_{lang}")):
            result = count(folder, lang, defs, inv, XL_HELD[lang])
            if result is None:
                print(f"{lang.upper():<5}{label:<34}{'not in archive':>11}")
                continue
            c, lemmas = result
            detail[f"{lang}:{label}"] = dict(c, words=len(lemmas))
            print(f"{lang.upper():<5}{label:<34}{c['instances']:>11,}{c['usable']:>9,}{len(lemmas):>8,}   "
                  f"{ {k: v for k, v in c.items() if k.startswith('skip')} }")
            if label.startswith("dev"):
                total, total_words = total + c["usable"], total_words + len(lemmas)
    print("\nThe dev splits are small but human-labelled. Translated SemCor is large but machine-made (silver).")
    return dict(language="IT/ES/NL", usable=total, words=total_words, license="CC BY-NC-SA (XL-WSD)",
                labels="dev: sense chosen by annotators; definitions are machine-translated WordNet glosses", counts=detail)

# ---------------------------------------------------------------------------
def check_finnish():
    import pandas as pd
    repo = WORK / "axolotl24_shared_task"
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", URLS["axolotl"], str(repo)], check=True)
    frame = pd.read_csv(repo / "data/finnish/axolotl.train.fi.tsv", sep="\t", quoting=3, dtype=str, keep_default_na=False)
    needed = {"word", "sense_id", "gloss", "example", "indices_target_token", "date"}
    assert needed <= set(frame.columns), f"Unexpected columns: {list(frame.columns)}"
    n_all = len(frame)
    frame = frame[frame["word"].map(norm).isin(FI_TRAIN_WORDS)]
    glosses = frame.groupby("word")["gloss"].nunique()
    c, words, samples, rng = Counter(), set(), [], random.Random(SEED)
    for row in frame.itertuples(index=False):
        match = re.fullmatch(r"(\d+):(\d+)", row.indices_target_token.strip())
        if not match or not row.gloss.strip() or not row.example.strip():
            c["skip_no_single_target_or_definition"] += 1
            continue
        a, b = int(match.group(1)), int(match.group(2))
        if not 0 <= a < b <= len(row.example) or not 15 <= len(row.example) <= 400:
            c["skip_bad_span_or_length"] += 1
            continue
        if glosses[row.word] < 2:
            c["skip_word_has_one_sense"] += 1
            continue
        if not re.fullmatch(r"\d{4}", row.date.strip()):
            c["skip_no_year"] += 1
            continue
        c["usable"] += 1
        words.add(norm(row.word))
        i = slot(c["usable"], 3, rng)
        if i is not None:
            samples[i:i + 1] = [(row.word, row.date, show(row.example, a, b), row.gloss[:150])]
    print(f"Training rows: {n_all:,} | rows for words in our frozen Finnish training split: {len(frame):,}")
    print(f"Usable (single target, definition, word has 2+ senses, year present): {c['usable']:,} over {len(words):,} words")
    print("Skipped:", {k: v for k, v in c.items() if k.startswith("skip")})
    for word, year, sentence, gloss in samples:
        print(f"\n  word: {word} ({year})\n  sentence: {sentence}\n  YES def : {gloss}")
    return dict(language="FI (historical, 1543-1810)", usable=c["usable"], words=len(words),
                license="repository GPL-3.0; source dictionary released as CC BY", labels="dictionary citations with their own sense", counts=dict(c))

# ---------------------------------------------------------------------------
run("1. ENGLISH: FEWS (Wiktionary quotations, many close senses)", check_fews)
run("2. ITALIAN: native Wiktionary (definitions written in Italian)", lambda: check_wiktionary("it", "Italian"))
run("3. SPANISH: native Wiktionary (definitions written in Spanish)", lambda: check_wiktionary("es", "Spanish"))
run("4. DUTCH: native Wiktionary (definitions written in Dutch)", lambda: check_wiktionary("nl", "Dutch"))
run("5. SWEDISH: Swedish Wiktionary dump (definitions written in Swedish)", check_swedish)
run("6. ITALIAN / SPANISH / DUTCH: XL-WSD training-side data", check_xlwsd)
run("7. FINNISH: AXOLOTL-24 Old Literary Finnish (historical text)", check_finnish)

section("SUMMARY: WHAT IS AVAILABLE FOR TRAINING ROUND 2")
print(f"{'source':<74}{'usable':>9}{'words':>8}   status")
for name, s in summary.items():
    print(f"{name:<74}{s.get('usable', 0):>9,}{s.get('words', 0):>8,}   {s['status']}")
print("\nAlready in round 1 (kept): Russian AXOLOTL 6,078 + English WiC-TSV 2,432 = 8,510 examples.")
print("'usable' = examples that can give one YES pair and one NO pair with a different sense of the same word.")
OUT = new_run(PROJECT, "step13_round2_feasibility")
save_json(OUT / "feasibility.json", dict(summary=summary, urls=URLS, seed=SEED,
          note="Counts only. No training data was built or saved. Downloads are on temporary Colab disk."))
failed = [n for n, s in summary.items() if s["status"] != "OK"]
print(f"\nSources checked: {len(summary)} | OK: {len(summary) - len(failed)} | failed: {len(failed)}")
print(f"Total time: {(time.perf_counter() - t_all) / 60:.1f} minutes. Saved: {OUT}")
print("Nothing was trained and no existing file was changed. Paste this whole output.")

In [ ]:
 import sys, re, json, gzip, bz2, math, time, zipfile, random, shutil, hashlib, subprocess, urllib.request
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
import numpy as np, torch
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer
from project_utils import new_run, save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

# ----- settings fixed before building (nothing here is tuned on any evaluation data) -----
QUOTA = {            # usages per language: (training, selection). Each usage gives 1 YES + 1 NO example.
    "EN": (750, 150), "IT": (600, 150), "ES": (600, 150), "NL": (600, 150), "SV": (600, 150), "FI": (400, 0)}
SIM_MAX = 0.93       # same rule as round 1: drop a NO pair when the two definitions mean nearly the same
SEED = 2027
MIN_SENTENCE, MAX_SENTENCE, MIN_DEF, MAX_DEF = 25, 400, 5, 300
LANG = {"EN": "English", "IT": "Italian", "ES": "Spanish", "NL": "Dutch", "SV": "Swedish", "FI": "Finnish", "RU": "Russian"}
SOURCE = {"EN": "FEWS", "IT": "Wiktionary-it (kaikki.org)", "ES": "Wiktionary-es (kaikki.org)",
          "NL": "Wiktionary-nl (kaikki.org)", "SV": "Wiktionary-sv (dump)", "FI": "AXOLOTL-24 Finnish"}
WORK = Path("/content/r2_sources")          # temporary Colab disk, NOT Google Drive
WORK.mkdir(exist_ok=True)
URLS = {
    "fews": "https://nlp.cs.washington.edu/fews/fews.zip",
    "wikt_it": "https://kaikki.org/itwiktionary/raw-wiktextract-data.jsonl.gz",
    "wikt_es": "https://kaikki.org/eswiktionary/raw-wiktextract-data.jsonl.gz",
    "wikt_nl": "https://kaikki.org/nlwiktionary/raw-wiktextract-data.jsonl.gz",
    "wikt_sv": "https://dumps.wikimedia.org/svwiktionary/latest/svwiktionary-latest-pages-articles.xml.bz2",
    "axolotl": "https://github.com/ltgoslo/axolotl24_shared_task.git",
}
ROUND1 = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
SPLITS = PROJECT / "outputs/runs/step07_frozen_splits/20261105T220410Z_0c2b5b52/splits.json"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
EN_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a/EN_wic_dev.jsonl"
norm = lambda w: (w or "").strip().casefold()

def download(url, dest, min_bytes=1_000_000):
    if dest.exists() and dest.stat().st_size >= min_bytes:
        return dest
    part = dest.with_name(dest.name + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "semeval2027-class-project/1.0 (research use)"})
    with urllib.request.urlopen(request, timeout=180) as response, open(part, "wb") as handle:
        shutil.copyfileobj(response, handle, 1 << 20)
    assert part.stat().st_size >= min_bytes, f"Download too small ({part.stat().st_size} bytes): {url}"
    part.replace(dest)
    return dest

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def messages(lang, word, definition, marked_text, when):          # word for word the round 1 / frozen prompt
    L = LANG[lang]
    return [{"role": "system", "content": f"You are an expert lexicographer of historical {L}."},
            {"role": "user", "content": (
                f"Target word (lemma): {word}\n"
                f"Sense definition ({L}): {definition}\n"
                f"Sentence from {when}: {marked_text}\n\n"
                "Does the word marked with <t> and </t> in the sentence express "
                "the sense given by the definition? Answer only Yes or No.")}]

def tidy(text):
    return " ".join((text or "").split())

def candidate(word, text, a, b, yes, others, when="modern times"):
    """One usable usage, or None. `others` = definitions of the other senses of the same word."""
    text = text.replace("\n", " ").replace("\t", " ")              # same length, so the target span stays valid
    yes = tidy(yes)
    if not (MIN_SENTENCE <= len(text) <= MAX_SENTENCE) or not (MIN_DEF <= len(yes) <= MAX_DEF):
        return None
    if not (0 <= a < b <= len(text)) or "<t>" in text or "</t>" in text:
        return None
    target = text[a:b]
    if target != target.strip() or any(ch.isspace() for ch in target) or not any(ch.isalpha() for ch in target):
        return None
    y = yes.casefold()
    nos = sorted({n for n in (tidy(o) for o in others)
                  if MIN_DEF <= len(n) <= MAX_DEF and n.casefold() != y and n.casefold() not in y and y not in n.casefold()})
    if not nos or not word or " " in word.strip():
        return None
    return dict(word=word.strip(), text=text, a=a, b=b, yes=yes, nos=nos, when=when)

# ---------------------------------------------------------------------------
section("0. HELD-OUT WORDS AND THE ROUND 1 TRAINING SET")
SV_DEV_WORDS = {"dumpa", "exportera", "fru", "fröken", "förort", "herre", "kvinna", "motiv", "skär", "suga"}
SV_STEMS = ("dump", "export", "fru", "fröken", "förort", "herr", "kvinn", "motiv", "skär", "sug")
splits = json.loads(SPLITS.read_text())
word_lists = splits["word_lists"]
EN_HELD = {norm(w) for w in word_lists.get("WiC_EN_validation", [])} | {norm(r["word"]) for r in read_jsonl(EN_EVAL)}
HELD = {"EN": EN_HELD, "SV": set(), "FI": set()}
for code in ("IT", "ES", "NL"):
    HELD[code] = {norm(r["word"]) for r in read_jsonl(XL_EVAL / f"{code}_xlwsd_test.jsonl")}
FI_TRAIN_WORDS = {norm(w) for w in word_lists.get("AXOLOTL_FI_train", [])}
r1_manifest = json.loads((ROUND1 / "manifest.json").read_text())
r1_path = ROUND1 / "pilot_train.jsonl"
assert hashlib.sha256(r1_path.read_bytes()).hexdigest() == r1_manifest["sha256"]["pilot_train.jsonl"], "Round 1 training file changed."
round1 = read_jsonl(r1_path)
assert len(round1) == 8510
ROUND1_WORDS = defaultdict(set)
for r in round1:
    ROUND1_WORDS[r["lang"]].add(norm(r["word"]))
print(f"Round 1 training file: hash PASS | {len(round1):,} examples {dict(Counter(r['lang'] for r in round1))} (kept unchanged)")
print(f"Held-out evaluation words: EN {len(HELD['EN']):,} | IT {len(HELD['IT']):,} | ES {len(HELD['ES']):,} | NL {len(HELD['NL']):,}")
print(f"Swedish: the 10 dev words are excluded everywhere; in Swedish Wiktionary any title containing {SV_STEMS} is excluded.")

def allowed(lang, word):
    w = norm(word)
    if w in SV_DEV_WORDS or w in HELD[lang]:
        return False
    return not (lang == "SV" and any(stem in w for stem in SV_STEMS))

# ---------------------------------------------------------------------------
section("1. READ EVERY SOURCE AND COLLECT USABLE USAGES")
pool, t0 = {}, time.perf_counter()

def add(by_word, lang, c):
    if c is not None and allowed(lang, c["word"]):
        by_word[norm(c["word"])].append(c)

# ----- English: FEWS -----
archive = zipfile.ZipFile(download(URLS["fews"], WORK / "fews.zip", 10_000_000))
names = archive.namelist()
senses, block = {}, {}
for line in archive.read(next(n for n in names if n.endswith("senses.txt"))).decode("utf-8").splitlines():
    line = line.strip()
    if not line:
        if "sense_id" in block:
            senses[block["sense_id"]] = block
        block = {}
        continue
    key, _, value = line.partition(":\t")
    block[key.rstrip(":")] = value.strip()
if "sense_id" in block:
    senses[block["sense_id"]] = block
family = defaultdict(list)
for sense_id, s in senses.items():
    if s.get("gloss"):
        family[sense_id.rsplit(".", 1)[0]].append(sense_id)
attribution = {}
quote_file = next((n for n in names if n.endswith("raw/quotations.txt")), None)
if quote_file:
    for line in archive.read(quote_file).decode("utf-8").splitlines():
        parts = line.rstrip("\n").split("\t")
        if len(parts) >= 3:
            attribution[(parts[0].strip(), parts[1].strip())] = parts[2]
def quote_year(text):
    m = re.search(r"\byear\s*=\s*(1[4-9]\d\d|20[0-2]\d)\b", text) or re.match(r"\W*(1[4-9]\d\d|20[0-2]\d)\b", text)
    return m.group(1) if m else None
by_word, with_year, year_examples = defaultdict(list), 0, []
for line in archive.read(next(n for n in names if n.endswith("train/train.txt"))).decode("utf-8").splitlines():
    parts = line.strip().split("\t")
    if len(parts) != 2 or parts[0].count("<WSD>") != 1 or parts[0].count("</WSD>") != 1 or parts[1] not in senses:
        continue
    sentence, label = parts
    a = sentence.index("<WSD>")
    plain = sentence.replace("<WSD>", "", 1)
    b = plain.index("</WSD>")
    plain = plain.replace("</WSD>", "", 1)
    source_note = attribution.get((sentence.strip(), label.strip()), "")
    year = quote_year(source_note)
    c = candidate(senses[label].get("word", ""), plain, a, b, senses[label].get("gloss", ""),
                  [senses[x]["gloss"] for x in family[label.rsplit(".", 1)[0]] if x != label], year or "modern times")
    if c is not None and allowed("EN", c["word"]):
        by_word[norm(c["word"])].append(c)
        with_year += year is not None
        if year and len(year_examples) < 3:
            year_examples.append((year, source_note[:70]))
pool["EN"] = by_word
n_en = sum(len(v) for v in by_word.values())
print(f"EN FEWS         : {n_en:>7,} usable usages over {len(by_word):>6,} words | real year found for {with_year:,} "
      f"({100 * with_year / max(n_en, 1):.0f}%), others say 'modern times'")
for year, text in year_examples:
    print(f"                    year check: {year} <- '{text}'")

# ----- Italian, Spanish, Dutch: native Wiktionary editions -----
TOKEN = re.compile(r"\w+")
def locate(text, offsets, forms):
    if isinstance(offsets, list) and len(offsets) == 1 and isinstance(offsets[0], (list, tuple)) and len(offsets[0]) == 2:
        a, b = offsets[0]
        if isinstance(a, int) and isinstance(b, int) and 0 <= a < b <= len(text):
            return a, b
    hits = [(m.start(), m.end()) for m in TOKEN.finditer(text) if m.group(0).casefold() in forms]
    return hits[0] if len(hits) == 1 else None
for code in ("it", "es", "nl"):
    lang, by_word = code.upper(), defaultdict(list)
    with gzip.open(download(URLS[f"wikt_{code}"], WORK / f"{code}wiktionary.jsonl.gz", 5_000_000), "rt", encoding="utf-8") as handle:
        for line in handle:
            try:
                entry = json.loads(line)
            except ValueError:
                continue
            if not isinstance(entry, dict) or entry.get("lang_code") != code:
                continue
            word = entry.get("word") or ""
            entry_senses = []
            for s in entry.get("senses") or []:
                glosses = [g.strip() for g in (s.get("glosses") or []) if isinstance(g, str) and g.strip()]
                if glosses and not (s.get("form_of") or s.get("alt_of") or "form-of" in (s.get("tags") or [])):
                    entry_senses.append((glosses[-1], [e for e in (s.get("examples") or []) if isinstance(e, dict)]))
            if len({g for g, _ in entry_senses}) < 2:
                continue
            forms = {norm(word)} | {norm(f.get("form")) for f in (entry.get("forms") or []) if isinstance(f, dict) and f.get("form")}
            forms = {f for f in forms if len(f) >= 2 and " " not in f}
            for gloss, examples in entry_senses:
                for e in examples:
                    raw = e.get("text") or ""
                    lead = len(raw) - len(raw.lstrip())
                    text = raw.strip()
                    offsets = e.get("bold_text_offsets")
                    if lead and isinstance(offsets, list):
                        offsets = [[o[0] - lead, o[1] - lead] for o in offsets
                                   if isinstance(o, (list, tuple)) and len(o) == 2 and all(isinstance(v, int) for v in o)]
                    span = locate(text, offsets, forms)
                    if span is not None:
                        add(by_word, lang, candidate(word, text, span[0], span[1], gloss, [g for g, _ in entry_senses if g != gloss]))
    pool[lang] = by_word
    print(f"{lang} Wiktionary   : {sum(len(v) for v in by_word.values()):>7,} usable usages over {len(by_word):>6,} words")

# ----- Swedish: Swedish Wiktionary dump -----
BOLD = "\x01"
def clean_wikitext(text):
    text = re.sub(r"<!--.*?-->|<ref[^>]*/>|<ref[^>]*>.*?</ref>", "", text, flags=re.S)
    previous = None
    while previous != text:
        previous, text = text, re.sub(r"\{\{[^{}]*\}\}", "", text)
    return re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", text)
def clean_definition(text):
    return re.sub(r"\s+", " ", clean_wikitext(text).replace("'''", "").replace("''", "")).strip(" .;:,")
def clean_example(text):
    text = re.sub(r"\s+", " ", clean_wikitext(text).replace("'''", BOLD).replace("''", "")).strip()
    if text.count(BOLD) != 2:
        return None
    start = text.index(BOLD)
    end = text.index(BOLD, start + 1) - 1
    return text.replace(BOLD, ""), start, end
def swedish_blocks(wikitext):
    match = re.search(r"^==\s*Svenska\s*==\s*$", wikitext, flags=re.M)
    if not match:
        return
    rest = wikitext[match.end():]
    nxt = re.search(r"^==[^=].*?==\s*$", rest, flags=re.M)
    pos, definitions = None, []
    for line in (rest[:nxt.start()] if nxt else rest).splitlines():
        heading = re.match(r"^===\s*([^=].*?)\s*===\s*$", line)
        if heading:
            if pos and definitions:
                yield definitions
            pos, definitions = heading.group(1), []
        elif line.startswith("#") and not line.startswith(("#:", "#*", "##")):
            definitions.append((line[1:], []))
        elif line.startswith("#:") and definitions and line[2:].strip().startswith("''"):
            definitions[-1][1].append(line[2:].strip())
    if pos and definitions:
        yield definitions
by_word = defaultdict(list)
with bz2.open(download(URLS["wikt_sv"], WORK / "svwiktionary.xml.bz2", 5_000_000), "rb") as handle:
    context = ET.iterparse(handle, events=("start", "end"))
    _, root = next(context)
    for event, element in context:
        if event != "end" or element.tag.rsplit("}", 1)[-1] != "page":
            continue
        title = ns = text = None
        for child in element.iter():
            tag = child.tag.rsplit("}", 1)[-1]
            if tag == "title":
                title = child.text
            elif tag == "ns":
                ns = child.text
            elif tag == "text":
                text = child.text
        root.clear()
        if ns != "0" or not title or not text or "Svenska" not in text:
            continue
        for definitions in swedish_blocks(text):
            cleaned = [(clean_definition(d), examples) for d, examples in definitions]
            cleaned = [(d, examples) for d, examples in cleaned if len(d) >= 3]
            if len({d for d, _ in cleaned}) < 2:
                continue
            for definition, examples in cleaned:
                for raw in examples:
                    parsed = clean_example(raw)
                    if parsed is not None:
                        add(by_word, "SV", candidate(title, parsed[0], parsed[1], parsed[2], definition,
                                                     [d for d, _ in cleaned if d != definition]))
pool["SV"] = by_word
print(f"SV Wiktionary   : {sum(len(v) for v in by_word.values()):>7,} usable usages over {len(by_word):>6,} words")

# ----- Finnish: AXOLOTL-24 Old Literary Finnish (real years) -----
import pandas as pd
repo = WORK / "axolotl24_shared_task"
if not repo.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", URLS["axolotl"], str(repo)], check=True)
frame = pd.read_csv(repo / "data/finnish/axolotl.train.fi.tsv", sep="\t", quoting=3, dtype=str, keep_default_na=False)
frame = frame[frame["word"].map(norm).isin(FI_TRAIN_WORDS)]
glosses_of = defaultdict(dict)
for row in frame.itertuples(index=False):
    if row.gloss.strip():
        glosses_of[row.word][row.sense_id] = row.gloss
# About 10% of the Finnish rows mark the wrong word (for example 'ja' for the headword 'arka'). Keep a row only when the
# marked word starts with the same letter as its headword, allowing old-spelling variants (c/k, w/v, i/j, t/d, p/b ...).
FI_GROUPS = ["ckqg", "vwuf", "ijy", "td", "pb", "sz", "aä", "oöå"]
FI_CLASS = {ch: i for i, group in enumerate(FI_GROUPS) for ch in group}
def fi_start(text):
    letters = re.sub(r"[^a-zåäö]", "", text.casefold())
    return FI_CLASS.get(letters[0], letters[0]) if letters else None
by_word, fi_dropped = defaultdict(list), Counter()
for row in frame.itertuples(index=False):
    match = re.fullmatch(r"(\d+):(\d+)", row.indices_target_token.strip())
    if not match or not re.fullmatch(r"\d{4}", row.date.strip()) or not row.gloss.strip():
        fi_dropped["no single target, year or definition"] += 1
        continue
    a, b = int(match.group(1)), int(match.group(2))
    if row.example[a:b] != row.orth:
        fi_dropped["position does not match the recorded word form"] += 1
        continue
    if fi_start(row.word) is None or fi_start(row.word) != fi_start(row.orth):
        fi_dropped["marked word does not match the headword"] += 1
        continue
    while b > a and row.example[b - 1] in ",.;:!?)":                 # the span sometimes includes punctuation
        b -= 1
    add(by_word, "FI", candidate(row.word, row.example, a, b, row.gloss,
                                 [g for sid, g in glosses_of[row.word].items() if sid != row.sense_id], row.date.strip()))
pool["FI"] = by_word
print(f"FI AXOLOTL      : {sum(len(v) for v in by_word.values()):>7,} usable usages over {len(by_word):>6,} words (years 1543-1810)")
print(f"                    Finnish rows dropped: {dict(fi_dropped)}")
print(f"[{time.perf_counter() - t0:.0f}s]")

# ---------------------------------------------------------------------------
section("2. PICK USAGES: SPLIT WORDS FIRST, THEN ONE USAGE PER WORD")
print("Selection words are drawn first; training words come from the rest, so no word is in both.")
print("Each usage gets its own definition (YES) and one random other sense of the same word (NO).\n")
picked = {}
for i, (lang, (n_train, n_sel)) in enumerate(QUOTA.items()):
    rng = random.Random(SEED + i)
    words = sorted(pool[lang])
    rng.shuffle(words)
    def draw(word_list, need):
        out = []
        for w in word_list:
            if len(out) >= math.ceil(need * 1.3) + 10:
                break
            c = rng.choice(pool[lang][w])
            out.append(dict(c, no=rng.choice(c["nos"]), lang=lang))
        return out
    sel_words = [w for w in words if w not in ROUND1_WORDS[lang]]
    selection = draw(sel_words, n_sel) if n_sel else []
    used = {norm(c["word"]) for c in selection}
    training = draw([w for w in words if w not in used], n_train)
    picked[lang] = dict(train=training, selection=selection)
    print(f"{lang}: drew {len(training):,} training and {len(selection):,} selection usages (30% extra, trimmed after the next step)")

# ---------------------------------------------------------------------------
section("3. REMOVE NO PAIRS WHOSE TWO DEFINITIONS MEAN NEARLY THE SAME (multilingual-e5-large)")
MODEL_ID = "intfloat/multilingual-e5-large"
revision = HfApi().model_info(MODEL_ID).sha
device = "cuda" if torch.cuda.is_available() else "cpu"
texts = sorted({d for groups in picked.values() for rows in groups.values() for c in rows for d in (c["yes"], c["no"])})
print(f"Device: {device} | {len(texts):,} definitions to compare"
      + ("" if device == "cuda" else " | on CPU this step takes about 15 to 25 minutes"))
encoder = SentenceTransformer(MODEL_ID, revision=revision, device=device)
vectors = dict(zip(texts, encoder.encode(["query: " + t for t in texts], batch_size=32, normalize_embeddings=True, show_progress_bar=True)))
del encoder
final, stats = {}, {}
print(f"\n{'lang':<5}{'split':<11}{'kept':>6}{'needed':>8}{'dropped (similar)':>19}   similarity percentiles 10/50/90 of kept pairs")
for lang, groups in picked.items():
    for split, rows in groups.items():
        need = QUOTA[lang][0 if split == "train" else 1]
        if not need:
            continue
        for c in rows:
            c["sim"] = float(np.clip(vectors[c["yes"]] @ vectors[c["no"]], -1, 1))
        kept = [c for c in rows if c["sim"] < SIM_MAX]
        dropped = len(rows) - len(kept)
        assert len(kept) >= need, f"{lang} {split}: only {len(kept)} usages left, {need} needed. STOP and paste this output."
        kept = kept[:need]
        final[(lang, split)] = kept
        stats[f"{lang}_{split}"] = dict(kept=len(kept), dropped_similar=dropped, drawn=len(rows))
        pct = np.percentile([c["sim"] for c in kept], [10, 50, 90]).round(3)
        print(f"{lang:<5}{split:<11}{len(kept):>6,}{need:>8,}{dropped:>19,}   {pct[0]:.3f} / {pct[1]:.3f} / {pct[2]:.3f}")

# ---------------------------------------------------------------------------
section("4. BUILD THE EXAMPLES AND CHECK THEM")
def rows_for(lang, split, usages):
    out = []
    for k, c in enumerate(usages):
        marked = c["text"][:c["a"]] + "<t>" + c["text"][c["a"]:c["b"]] + "</t>" + c["text"][c["b"]:]
        for label, definition in ((1, c["yes"]), (0, c["no"])):
            out.append(dict(id=f"r2_{lang}_{split}_{k:05d}|{'yes' if label else 'no'}", source=SOURCE[lang], lang=lang,
                            split=split, word=c["word"], label=label, meaning_sim=round(c["sim"], 4),
                            messages=messages(lang, c["word"], definition, marked, c["when"])))
    return out
new_train = [r for (lang, split), u in final.items() if split == "train" for r in rows_for(lang, split, u)]
new_sel = [r for (lang, split), u in final.items() if split == "selection" for r in rows_for(lang, split, u)]
train = round1 + new_train
random.Random(SEED).shuffle(train)

def sentence_part(r):
    return r["messages"][1]["content"].split("Sentence from ", 1)[1].rsplit("\n\nDoes the word marked", 1)[0]
def words_of(rows, lang):
    return {norm(r["word"]) for r in rows if r["lang"] == lang}
checks = {}
train_by_id = {r["id"]: r for r in train}
checks["round 1 examples all present and unchanged"] = all(train_by_id.get(r["id"]) == r for r in round1) and \
    len(train) == len(round1) + len(new_train)
checks["all example IDs unique (training + selection)"] = len({r["id"] for r in train + new_sel}) == len(train) + len(new_sel)
checks["every new example has exactly one <t> target"] = all(
    sentence_part(r).count("<t>") == 1 and sentence_part(r).count("</t>") == 1 for r in new_train + new_sel)
checks["new examples: one YES and one NO per usage"] = all(
    sum(r["label"] for r in rows) * 2 == len(rows) for rows in (new_train, new_sel))
checks["10 Swedish dev words: not in training or selection"] = not any(norm(r["word"]) in SV_DEV_WORDS for r in train + new_sel)
checks["Swedish Wiktionary: no dev-word stems or compounds"] = not any(
    stem in norm(r["word"]) for r in new_train + new_sel if r["lang"] == "SV" for stem in SV_STEMS)
checks["no held-out evaluation word (EN, IT, ES, NL)"] = not any(
    norm(r["word"]) in HELD[r["lang"]] for r in new_train + new_sel)
checks["selection words never appear in training"] = all(
    not (words_of(new_sel, lang) & words_of(train, lang)) for lang in QUOTA)
checks["prompts end with the question (no answers inside)"] = all(
    r["messages"][-1]["role"] == "user" and {m["role"] for m in r["messages"]} == {"system", "user"} for r in train + new_sel)
checks["quotas met exactly"] = all(len(final[(lang, "train")]) == q[0] and (not q[1] or len(final[(lang, "selection")]) == q[1])
                                   for lang, q in QUOTA.items())
for name, ok in checks.items():
    print(f"  {name:<54} {'PASS' if ok else 'FAIL'}")
assert all(checks.values()), "A check failed: STOP here and paste the output. Nothing was saved."

# ---------------------------------------------------------------------------
section("5. EYE CHECK: 3 random training usages per new language")
print("Check: does the YES definition fit the marked word, and is the NO definition a clearly different sense?\n")
for lang in QUOTA:
    for c in random.Random(SEED).sample(final[(lang, "train")], 3):
        marked = c["text"][:c["a"]] + "<t>" + c["text"][c["a"]:c["b"]] + "</t>" + c["text"][c["b"]:]
        print(f"[{lang}] word: {c['word']} | from {c['when']} | definition similarity {c['sim']:.3f}")
        print(f"   sentence: {marked[:240]}\n   YES def : {c['yes'][:150]}\n   NO def  : {c['no'][:150]}\n")

# ---------------------------------------------------------------------------
section("6. SAVE THE ROUND 2 DATASET")
OUT = new_run(PROJECT, "step13_round2_dataset")
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as handle:
        for r in rows:
            handle.write(json.dumps(r, ensure_ascii=False) + "\n")
    return hashlib.sha256(path.read_bytes()).hexdigest()
h_train = write_jsonl(OUT / "round2_train.jsonl", train)
h_sel = write_jsonl(OUT / "round2_selection_new_languages.jsonl", new_sel)
print(f"{'language':<10}{'examples':>9}{'YES':>7}{'NO':>7}   prompt length median / max (characters)")
for lang in sorted({r["lang"] for r in train}):
    rows = [r for r in train if r["lang"] == lang]
    chars = sorted(len(r["messages"][1]["content"]) for r in rows)
    print(f"{lang:<10}{len(rows):>9,}{sum(r['label'] for r in rows):>7,}{sum(1 - r['label'] for r in rows):>7,}   "
          f"{chars[len(chars) // 2]} / {chars[-1]}")
years = sum(1 for r in new_train if "Sentence from modern times" not in r["messages"][1]["content"])
print(f"\nTraining examples with a real year in the prompt: {years:,} of {len(train):,} (round 1 had none)")
save_json(OUT / "manifest.json", {
    "round1_train": str(r1_path), "round1_sha256": r1_manifest["sha256"]["pilot_train.jsonl"],
    "round1_selection": str(ROUND1 / "pilot_selection_ru_dev.jsonl"),
    "quota_usages_train_selection": QUOTA, "sources": SOURCE, "urls": URLS,
    "licenses": {"FEWS": "CC BY-NC 4.0", "Wiktionary (it, es, nl, sv)": "CC BY-SA", "AXOLOTL-24 Finnish": "repository GPL-3.0; source dictionary CC BY"},
    "filters": {"meaning_similarity_max": SIM_MAX, "similarity_model": MODEL_ID, "similarity_revision": revision,
                "sentence_chars": [MIN_SENTENCE, MAX_SENTENCE], "definition_chars": [MIN_DEF, MAX_DEF],
                "one_usage_per_word": True, "single_token_target": True,
                "no_choice": "random other sense of the same word (same construction as round 1 Russian)"},
    "held_out": {"swedish_dev_words": sorted(SV_DEV_WORDS), "swedish_stems_excluded": list(SV_STEMS),
                 "EN": "WiC-TSV dev words", "IT/ES/NL": "XL-WSD test lemmas (step11 evaluation sets)"},
    "not_used": "XL-WSD dev splits (only 192 usable usages) and machine-translated SemCor (silver labels and silver definitions)",
    "stats": stats, "counts": {"train": len(train), "new_train": len(new_train), "selection_new": len(new_sel)},
    "sha256": {"round2_train.jsonl": h_train, "round2_selection_new_languages.jsonl": h_sel},
    "checks": checks, "seed": SEED})

section("SUMMARY")
steps = math.ceil(len(train) / 16)
print(f"Round 2 training set : {len(train):,} examples = round 1 ({len(round1):,}) + new ({len(new_train):,})   sha256 {h_train[:12]}...")
print(f"New selection set    : {len(new_sel):,} examples in EN, IT, ES, NL, SV (words never used in training)   sha256 {h_sel[:12]}...")
print(f"Training estimate    : {steps:,} steps x 16.3 s = about {steps * 16.3 / 3600:.1f} hours on the A100")
print(f"Saved: {OUT}")
print("No model was trained. Round 1 files are unchanged. Paste this whole output.")

In [ ]:
"""ROUND 2 TRAINING. Paste into ONE cell of notebook 12 on an A100 runtime and run it.
Fresh LoRA adapter, one pass over the round 2 training set, same recipe as round 1.
If Colab disconnects: reconnect to an A100 and run this same cell again. It resumes by itself."""
import sys, gc, json, math, time, uuid, random, shutil, hashlib, subprocess, importlib
from contextlib import nullcontext
from importlib.metadata import version
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
importlib.invalidate_caches()
import numpy as np, torch, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from peft import LoraConfig, PeftModel, get_peft_model
from peft.import_utils import is_torchao_available
from project_utils import new_run, save_json
from semeval_task3.data import load_dataset

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

DEVICE = "cuda"
MODEL_ID = "google/gemma-4-31B-it"
REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"
R2 = PROJECT / "outputs/runs/step13_round2_dataset/20261114T214116Z_acd7af91"
R1 = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
POINTER = R2 / "round2_lora_run_pointer.json"
SEED, EFFECTIVE_BATCH, LR = 2027, 16, 1e-4
SAVE_EVERY = 50
N_TARGET_LAYERS = 230
GRID = tuple([-1000.0] + np.round(np.arange(-100, 100.001, 0.25), 2).tolist() + [1000.0])
SV_DEV_WORDS = {"dumpa", "exportera", "fru", "fröken", "förort", "herre", "kvinna", "motiv", "skär", "suga"}

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

# ---------------------------------------------------------------------------
section("1. CHECK EVERY INPUT FILE BEFORE TOUCHING THE MODEL")
assert torch.cuda.is_available() or DEVICE != "cuda", "No GPU: Runtime > Change runtime type > A100."
is_torchao_available.cache_clear()
assert is_torchao_available() is False, "torchao is still visible to peft: Runtime > Restart session, then run this cell again."
m2 = json.loads((R2 / "manifest.json").read_text())
m1 = json.loads((R1 / "manifest.json").read_text())
train_path = R2 / "round2_train.jsonl"
sel_new_path = R2 / "round2_selection_new_languages.jsonl"
sel_ru_path = R1 / "pilot_selection_ru_dev.jsonl"
assert digest(train_path) == m2["sha256"][train_path.name], "Round 2 training file changed."
assert digest(sel_new_path) == m2["sha256"][sel_new_path.name], "Round 2 selection file changed."
assert digest(sel_ru_path) == m1["sha256"][sel_ru_path.name], "Russian selection file changed."
training = read_rows(train_path)
selection = read_rows(sel_ru_path) + read_rows(sel_new_path)
assert len(training) == 15610 and len(selection) == 2946 + 1500
for rows in (training, selection):
    assert len({str(r["id"]) for r in rows}) == len(rows)
    for r in rows:
        assert int(r["label"]) in (0, 1)
        assert r["messages"] and r["messages"][-1]["role"] == "user"
        assert all(m["role"] in ("system", "user") for m in r["messages"])
assert not any(r["word"].strip().casefold() in SV_DEV_WORDS for r in training + selection), "A Swedish dev word is in the data."
sel_words = {(r["lang"], r["word"].strip().casefold()) for r in selection}
assert not sel_words & {(r["lang"], r["word"].strip().casefold()) for r in training}, "Selection words overlap training words."
SEL_LANGS = sorted({r["lang"] for r in selection})
SEL_Y = {lang: np.array([int(r["label"]) for r in selection if r["lang"] == lang]) for lang in SEL_LANGS}
SEL_IDX = {lang: np.array([i for i, r in enumerate(selection) if r["lang"] == lang]) for lang in SEL_LANGS}
print(f"Training file   : hash PASS | {len(training):,} examples | "
      f"{dict(sorted({lang: sum(r['lang'] == lang for r in training) for lang in {r['lang'] for r in training}}.items()))}")
print(f"Selection files : hash PASS | {len(selection):,} examples | { {lang: len(SEL_Y[lang]) for lang in SEL_LANGS} }")

# Evaluation inputs (scored at the very end). Their labels are never used in this cell.
enru_manifest = json.loads((ENRU_EVAL / "manifest.json").read_text())
xl_manifest = json.loads((XL_EVAL / "manifest.json").read_text())
eval_sets = {}
for name in ("EN_wic_dev", "RU_axolotl_test_derived"):
    assert digest(ENRU_EVAL / f"{name}.jsonl") == enru_manifest["output_sha256"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(ENRU_EVAL / f"{name}.jsonl")
for lang in ("IT", "ES", "NL"):
    name = f"{lang}_xlwsd_test"
    assert digest(XL_EVAL / f"{name}.jsonl") == xl_manifest["files"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(XL_EVAL / f"{name}.jsonl")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=False)
eval_sets["SV_dev"] = [dict(id="|".join(u.key), messages=[
    dict(role="system", content="You are an expert lexicographer of historical Swedish."),
    dict(role="user", content=(f"Target word (lemma): {u.word}\n"
                               f"Sense definition (Swedish): {u.definition}\n"
                               f"Sentence from {u.year}: {u.marked_text()}\n\n"
                               "Does the word marked with <t> and </t> in the sentence express the sense given by the definition? "
                               "Answer only Yes or No."))]) for u in sv.usages]
assert len(eval_sets["SV_dev"]) == 1983
for name, rows in eval_sets.items():
    assert all(r["messages"][-1]["role"] == "user" for r in rows) and len({str(r["id"]) for r in rows}) == len(rows)
print(f"Evaluation sets : hash PASS | { {name: len(rows) for name, rows in eval_sets.items()} }")

random.Random(SEED).shuffle(training)
TOTAL = math.ceil(len(training) / EFFECTIVE_BATCH)
MILESTONES = sorted({math.ceil(TOTAL * 0.25), math.ceil(TOTAL * 0.5), TOTAL})
WARMUP = math.ceil(0.05 * TOTAL)
config = dict(model=MODEL_ID, revision=REVISION, seed=SEED, epochs=1, examples=len(training), micro_batch=1,
              effective_batch=EFFECTIVE_BATCH, learning_rate=LR, warmup_steps=WARMUP,
              schedule="linear warmup then linear decay", steps=TOTAL, milestones=MILESTONES,
              r=16, alpha=32, dropout=0.05, target="language-model q/k/v/o", base_precision="BF16, no quantization",
              loss="full-vocabulary cross entropy at next answer token", adapter_start="fresh (not continued from round 1)",
              train_sha256=digest(train_path), selection_sha256=[digest(sel_ru_path), digest(sel_new_path)],
              checkpoint_selection="highest mean over selection languages of macro-F1 at one shared cutoff; ties -> earlier step",
              cutoff_grid=[GRID[1], GRID[-2], 0.25],
              packages={p: version(p) for p in ("torch", "transformers", "peft")})
if POINTER.exists():
    FULL_RUN = Path(json.loads(POINTER.read_text())["run_directory"])
    assert json.loads((FULL_RUN / "config.json").read_text()) == json.loads(json.dumps(config)), (
        "The run settings changed since this run started. Do not resume with different settings: paste this message.")
    print(f"Existing run found: it will RESUME. {FULL_RUN}")
else:
    FULL_RUN = new_run(PROJECT, "step14_round2_lora")
    save_json(FULL_RUN / "config.json", config)
    save_json(POINTER, {"run_directory": str(FULL_RUN)})
    print(f"New run: {FULL_RUN}")
print(f"{TOTAL} updates of {EFFECTIVE_BATCH} examples | selection checks at updates {MILESTONES} | warm-up {WARMUP} updates")

# ---------------------------------------------------------------------------
section("2. GEMMA 4 31B (bf16)")
if "full_model" in globals() or "gemma_model" in globals():
    print("Gemma is already loaded in this session: reusing it.")
else:
    from transformers import AutoProcessor, AutoModelForMultimodalLM
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 2**30:.0f} GiB | transformers {version('transformers')}")
    assert props.total_memory / 2**30 > 70, "Gemma 31B training needs the A100 80 GB (or G4)."
    print("Loading Gemma (10 to 15 minutes; it downloads about 60 GB) ...", flush=True)
    gemma_processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
    gemma_tokenizer = gemma_processor.tokenizer
    gemma_model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, revision=REVISION, dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
    print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
gemma_tokenizer.padding_side = "left"
assert gemma_tokenizer.encode("Yes", add_special_tokens=False) == [10784]
assert gemma_tokenizer.encode("No", add_special_tokens=False) == [3771]
YES_ID, NO_ID = 10784, 3771

if (FULL_RUN / "SCORING_COMPLETE.json").exists():
    section("THIS RUN IS ALREADY COMPLETE")
    print("Nothing to do. Results:", FULL_RUN)
    print("Next: run the decision cell.")
else:
    # ---------------------------------------------------------------------------
    section("3. ADAPTER: FRESH START OR RESUME")
    current = globals().get("full_model")
    if current is None:
        current = globals().get("gemma_model")
    base = current.unload() if isinstance(current, PeftModel) else current     # unload() removes adapters without merging
    assert base.config._name_or_path == MODEL_ID, "Unexpected base model."
    assert not any("lora_" in n for n, _ in base.named_parameters())
    for name in ("full_optimizer", "params", "full_model", "gemma_model", "model"):
        globals().pop(name, None)
    del current
    gc.collect()
    torch.cuda.empty_cache()
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    targets = [n for n, m in base.named_modules() if isinstance(m, torch.nn.Linear) and "language_model" in n.split(".")
               and n.split(".")[-1] in ("q_proj", "k_proj", "v_proj", "o_proj")]
    assert len(targets) == N_TARGET_LAYERS, f"Unexpected number of attention layers: {len(targets)}"

    latest_file = FULL_RUN / "latest.json"
    resume_state = None
    if latest_file.exists():
        latest = Path(json.loads(latest_file.read_text())["directory"])
        full_model = PeftModel.from_pretrained(base, str(latest / "adapter"), is_trainable=True)
        resume_state = torch.load(latest / "training_state.pt", map_location="cpu", weights_only=False)   # our own file
        step_done, train_log = resume_state["step"], resume_state["log"]
        print(f"Resuming after update {step_done} of {TOTAL}.")
    else:
        full_model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=targets, bias="none"))
        step_done, train_log = 0, []
        assert all(torch.count_nonzero(p).item() == 0 for n, p in full_model.named_parameters() if "lora_B" in n)
        print("Fresh adapter attached: PASS (it starts from zero, not from the round 1 adapter).")
    full_model.config.use_cache = False
    full_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    params = [p for p in full_model.parameters() if p.requires_grad]
    assert params and all("lora_" in n for n, p in full_model.named_parameters() if p.requires_grad)
    print(f"Trainable parameters: {sum(p.numel() for p in params) / 1e6:.1f}M (LoRA weights only: PASS)")
    full_optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    if resume_state:
        full_optimizer.load_state_dict(resume_state["optimizer"])
        random.setstate(resume_state["python_rng"])
        np.random.set_state(resume_state["numpy_rng"])
        torch.set_rng_state(resume_state["torch_rng"])
        if resume_state.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(resume_state["cuda_rng"])
        del resume_state

    def encode(batch):
        inputs = gemma_processor.apply_chat_template(
            [r["messages"] for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True, enable_thinking=False,
            processor_kwargs={"padding": True, "truncation": False}).to(DEVICE)
        assert inputs["attention_mask"][:, -1].all(), "Padding is not on the left."
        return inputs

    def save_resume(step):
        folder = FULL_RUN / f"resume_{step:04d}_{uuid.uuid4().hex[:8]}"
        folder.mkdir()
        full_model.save_pretrained(folder / "adapter", safe_serialization=True)
        torch.save(dict(step=step, log=train_log, optimizer=full_optimizer.state_dict(), python_rng=random.getstate(),
                        numpy_rng=np.random.get_state(), torch_rng=torch.get_rng_state(),
                        cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None),
                   folder / "training_state.pt")
        previous = json.loads(latest_file.read_text())["directory"] if latest_file.exists() else None
        save_json(latest_file, {"directory": str(folder), "step": step})
        if previous:                                   # delete only this run's previous rolling checkpoint
            previous = Path(previous)
            if previous.parent == FULL_RUN and previous.name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
        save_json(FULL_RUN / "training_log.json", train_log)

    def score_rows(rows, destination, model_tag):
        """Resumable scoring. Labels never enter the model. Saves to Drive every 100 batches."""
        identity = hashlib.sha256(json.dumps([(str(r["id"]), r["messages"]) for r in rows],
                                             ensure_ascii=False, sort_keys=True).encode()).hexdigest()
        if destination.exists():
            saved = json.loads(destination.read_text())
            assert saved["input_sha256"] == identity and saved["model_tag"] == model_tag, f"{destination.name}: saved scores belong to other inputs."
        else:
            saved = dict(input_sha256=identity, model_tag=model_tag, scores={})
        pending = [r for r in rows if str(r["id"]) not in saved["scores"]]
        full_model.eval()
        t0, n_batches = time.perf_counter(), math.ceil(len(pending) / 2)
        try:
            with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.bfloat16):
                for b, i in enumerate(range(0, len(pending), 2), 1):
                    batch = pending[i:i + 2]
                    logits = full_model(**encode(batch), use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
                    assert torch.isfinite(logits).all()
                    margins = (logits[:, YES_ID] - logits[:, NO_ID]).tolist()
                    masses = (logits.log_softmax(-1)[:, [YES_ID, NO_ID]].exp().sum(-1)).tolist()
                    for r, margin, mass in zip(batch, margins, masses):
                        saved["scores"][str(r["id"])] = dict(margin=float(margin), yes_no_mass=float(mass))
                    del logits
                    if b % 100 == 0:
                        save_json(destination, saved)
                    if b % 500 == 0 or b == n_batches:
                        print(f"      {model_tag}: {b:,}/{n_batches:,} batches ({(time.perf_counter() - t0) / 60:.1f} min)", flush=True)
        finally:
            save_json(destination, saved)
        assert set(saved["scores"]) == {str(r["id"]) for r in rows}
        return [saved["scores"][str(r["id"])]["margin"] for r in rows]

    def macro_f1(y, pred):
        total = 0.0
        for c in (1, 0):
            denominator = np.sum(y == c) + np.sum(pred == c)
            total += 2 * np.sum((y == c) & (pred == c)) / denominator if denominator else 0.0
        return total / 2

    def selection_metrics(margins):
        """Official-style metric on the selection set: macro-F1 per language, one shared cutoff, mean over languages."""
        s = np.asarray(margins, dtype=float)
        assert np.isfinite(s).all() and np.max(np.abs(s)) < 100, f"Selection scores outside the cutoff grid: {s.min():.1f} to {s.max():.1f}"
        best = None
        for t in GRID:                                 # ascending, so ties keep the lower cutoff
            per = {lang: float(macro_f1(SEL_Y[lang], (s[SEL_IDX[lang]] >= t).astype(int))) for lang in SEL_LANGS}
            mean = float(np.mean(list(per.values())))
            if best is None or mean > best["score"] + 1e-12:
                best = dict(score=mean, threshold=float(t), per_language=per)
        best["auc"] = {lang: float(roc_auc_score(SEL_Y[lang], s[SEL_IDX[lang]])) for lang in SEL_LANGS}
        return best

    def milestone(step):
        folder = FULL_RUN / f"checkpoint_{step:04d}"
        folder.mkdir(exist_ok=True)
        if not (folder / "adapter" / "adapter_config.json").exists():
            full_model.save_pretrained(folder / "adapter", safe_serialization=True)
        result_path = folder / "selection_metrics.json"
        if not result_path.exists():
            margins = score_rows(selection, folder / "selection_scores.json", f"selection_step_{step}")
            save_json(result_path, dict(step=step, **selection_metrics(margins)))
        result = json.loads(result_path.read_text())
        print(f"   SELECTION at update {step}: mean macro-F1 {result['score']:.4f} (cutoff {result['threshold']:+.2f}) | "
              + " ".join(f"{lang} {v:.3f}" for lang, v in result["per_language"].items()), flush=True)
        full_model.train()

    # A 4-example scoring test now, so a scoring problem shows up in the first minute and not after an hour.
    smoke = score_rows(selection[:4], Path("/content") / f"smoke_{uuid.uuid4().hex[:8]}.json", "smoke")
    print(f"Scoring test on 4 examples: PASS (scores {[round(x, 2) for x in smoke]})")

    # ---------------------------------------------------------------------------
    section(f"4. TRAINING: updates {step_done + 1} to {TOTAL}")
    print("Loss should fall from above 1 to roughly 0.1-0.5 and jump around: each update sees only 16 examples.")
    print("'accuracy' is on TRAINING examples. It is not an evaluation result.\n")
    if step_done in MILESTONES:                        # a selection check interrupted by a disconnect is finished first
        milestone(step_done)
    full_model.train()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    session_start = time.perf_counter()
    for step in range(step_done + 1, TOTAL + 1):
        group = training[(step - 1) * EFFECTIVE_BATCH:step * EFFECTIVE_BATCH]
        assert group
        scale = step / WARMUP if step <= WARMUP else (TOTAL - step + 1) / (TOTAL - WARMUP)
        for pg in full_optimizer.param_groups:
            pg["lr"] = LR * scale
        full_optimizer.zero_grad(set_to_none=True)
        loss_sum, correct = 0.0, 0
        for r in group:
            answer = torch.tensor([YES_ID if int(r["label"]) else NO_ID], device=DEVICE)
            with torch.autocast(DEVICE, dtype=torch.bfloat16):
                logits = full_model(**encode([r]), use_cache=False, logits_to_keep=1).logits[:, -1, :]
            loss = F.cross_entropy(logits.float(), answer)
            assert torch.isfinite(loss), "Non-finite loss: stop. The last saved checkpoint is intact; paste this message."
            (loss / len(group)).backward()             # also correct for the final, smaller group
            loss_sum += float(loss.detach())
            correct += int((logits[0, YES_ID] > logits[0, NO_ID]).item() == bool(int(r["label"])))
            del logits, loss, answer
        norm = float(torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True))
        full_optimizer.step()
        train_log.append(dict(step=step, loss=loss_sum / len(group), batch_accuracy=correct / len(group),
                              grad_norm=norm, lr=LR * scale, examples=len(group)))
        if step == 1 or step % 20 == 0 or step in MILESTONES:
            done = step - step_done
            minutes = (time.perf_counter() - session_start) / 60
            print(f"Update {step:>4}/{TOTAL}: loss {loss_sum / len(group):.4f} | accuracy {correct / len(group):.2f} | "
                  f"{minutes:.0f} min this session | about {minutes / done * (TOTAL - step):.0f} min of training left", flush=True)
        if step % SAVE_EVERY == 0 or step in MILESTONES:
            save_resume(step)
        if step in MILESTONES:
            milestone(step)
    if torch.cuda.is_available():
        print(f"Peak GPU memory this session: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB of 80")

    # ---------------------------------------------------------------------------
    section("5. CHOOSE THE CHECKPOINT (selection sets only: Russian, English, Italian, Spanish, Dutch, Swedish Wiktionary)")
    candidates = [json.loads((FULL_RUN / f"checkpoint_{s:04d}/selection_metrics.json").read_text()) for s in MILESTONES]
    best = max(candidates, key=lambda x: (x["score"], -x["step"]))
    print(f"{'update':>7}{'mean macro-F1':>15}{'cutoff':>8}   per language")
    for c in candidates:
        print(f"{c['step']:>7}{c['score']:>15.4f}{c['threshold']:>8.2f}   " + " ".join(f"{k} {v:.3f}" for k, v in c["per_language"].items())
              + ("   <- chosen" if c["step"] == best["step"] else ""))
    save_json(FULL_RUN / "selection_decision.json", dict(candidates=candidates, selected=best,
              rule="highest mean macro-F1 over selection languages; ties -> earlier update; no Swedish dev data used"))
    full_optimizer.zero_grad(set_to_none=True)
    del full_optimizer, params
    gc.collect()
    torch.cuda.empty_cache()
    base = full_model.unload()
    full_model = PeftModel.from_pretrained(base, str(FULL_RUN / f"checkpoint_{best['step']:04d}" / "adapter"), is_trainable=False)
    full_model.eval()
    adapter_file = FULL_RUN / f"checkpoint_{best['step']:04d}" / "adapter" / "adapter_model.safetensors"
    print(f"Chosen adapter reloaded from disk: update {best['step']} | sha256 {digest(adapter_file)[:16]}...")

    # ---------------------------------------------------------------------------
    section("6. SCORE THE CHOSEN CHECKPOINT ON EVERY EVALUATION SET (about 17 minutes)")
    print("Only scores are saved here. Labels are not read, and no metric is computed in this cell.")
    (FULL_RUN / "eval_scores").mkdir(exist_ok=True)
    for name, rows in eval_sets.items():
        score_rows(rows, FULL_RUN / "eval_scores" / f"{name}.json", f"chosen_{best['step']}_{name}")
        print(f"   {name}: {len(rows):,} scores saved", flush=True)
    save_json(FULL_RUN / "SCORING_COMPLETE.json", dict(
        selected=best, run_directory=str(FULL_RUN), adapter=str(adapter_file.parent), adapter_sha256=digest(adapter_file),
        eval_scores={name: f"eval_scores/{name}.json" for name in eval_sets},
        status="training and scoring complete; adoption is decided by the decision cell, not here"))

    section("SUMMARY")
    print(f"Round 2 training finished. Chosen checkpoint: update {best['step']} of {TOTAL} (selection mean macro-F1 {best['score']:.4f}).")
    print(f"Everything is saved in: {FULL_RUN}")
    print("The current primary system (round 1, Swedish 0.753) is untouched.")
    print("Next: paste this output, then run the decision cell (1 minute, no GPU needed).")

In [ ]:
"""ROUND 2 DECISION. Run AFTER the training cell has finished (no GPU needed, about 1 minute).
It compares the round 2 model with the current primary system (round 1) using the rule fixed before training."""
import sys, json, subprocess, importlib.util
from collections import defaultdict
from pathlib import Path
import numpy as np
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
from sklearn.metrics import roc_auc_score
from project_utils import save_json, read_scores, frozen
from semeval_task3.data import load_dataset
import semeval_task3.cv as cv
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score as positive_score, word_bootstrap

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

R2_DATA = PROJECT / "outputs/runs/step13_round2_dataset/20261114T214116Z_acd7af91"
R1_RUN = PROJECT / "outputs/runs/step09_lora_full_pilot/20261105T232211Z_6b04a51b"
ORIG_SV = PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67/gemma_raw_scores.json"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
ENRU_BASE = PROJECT / "outputs/runs/step08_gemma_unchanged_baseline/20261105T224131Z_63bba2d9"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
XL_SCORES = PROJECT / "outputs/runs/step11_xlwsd_scores/adapter_6f3d236ac50e"
SCORER_DIR, SCORER_COMMIT = Path("/content/official-scorer-latest"), "0e69238"
ROUND1_SV_MACRO = 0.7529025040663228          # current primary system, from the frozen config
GRID = tuple([-1000.0] + np.round(np.arange(-100, 100.001, 0.25), 2).tolist() + [1000.0])

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

# ---------------------------------------------------------------------------
section("1. LOAD THE ROUND 2 RUN AND THE OFFICIAL SCORER")
R2_RUN = Path(json.loads((R2_DATA / "round2_lora_run_pointer.json").read_text())["run_directory"])
assert (R2_RUN / "SCORING_COMPLETE.json").exists(), "The training cell has not finished yet. Run this cell after it prints its SUMMARY."
done = json.loads((R2_RUN / "SCORING_COMPLETE.json").read_text())
choice = json.loads((R2_RUN / "selection_decision.json").read_text())
print(f"Round 2 run        : {R2_RUN.name}")
print(f"Chosen checkpoint  : update {done['selected']['step']} (selection mean macro-F1 {done['selected']['score']:.4f})")
print("Checkpoint choices : " + " | ".join(f"update {c['step']}: {c['score']:.4f}" for c in choice["candidates"]))

def macro_score(gold, predictions):
    """Official metric: per word, the mean of Yes-F1 and No-F1; then the mean over words (same report format as our harness)."""
    report = positive_score(gold, predictions)
    languages = defaultdict(list)
    for r in report["per_word"]:
        tp, fp, fn = r["tp"], r["fp"], r["fn"]
        tn = r["n"] - tp - fp - fn
        parts = []
        if tp + fp + fn > 0:
            parts.append(2 * tp / (2 * tp + fp + fn))
        if tn + fp + fn > 0:
            parts.append(2 * tn / (2 * tn + fn + fp))
        r["positive_f1"], r["f1"] = r["f1"], sum(parts) / len(parts)
        languages[r["language"]].append(r["f1"])
    report["per_language"] = {k: sum(v) / len(v) for k, v in languages.items()}
    report["average"] = sum(report["per_language"].values()) / len(report["per_language"])
    return report

if not SCORER_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/ChangeIsKey/semeval2027-task3.git", str(SCORER_DIR)], check=True)
subprocess.run(["git", "-C", str(SCORER_DIR), "checkout", "-q", SCORER_COMMIT], check=True)
assert "average='macro'" in (SCORER_DIR / "evaluation.py").read_text(), "Scorer is not the macro version."
sys.path.insert(0, str(SCORER_DIR))
spec = importlib.util.spec_from_file_location("official_eval_latest", SCORER_DIR / "evaluation.py")
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
def official_check(gold, predictions):
    return official.compute_f1_score({f"{k[1]};{k[2]}": int(v) for k, v in gold.items()},
                                     {f"{k[1]};{k[2]}": int(predictions[k]) for k in gold}, "check")
print(f"Official scorer    : commit {SCORER_COMMIT}, macro-F1 version confirmed")

# ---------------------------------------------------------------------------
section("2. SWEDISH DEV, OFFICIAL METRIC (nested leave-one-word-out: each word's cutoff chosen without its labels)")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = sv.require_gold()
def margins_by_key(path, tag):
    saved = json.loads(Path(path).read_text())
    assert saved["model_tag"] == tag, f"Unexpected scores in {path}: {saved['model_tag']}"
    return {u.key: float(saved["scores"]["|".join(u.key)]["margin"]) for u in sv.usages}
sv_scores = {"original": read_scores(ORIG_SV, sv),
             "round 1": margins_by_key(R1_RUN / "SV_scores.json", "chosen_532_SV"),
             "round 2": margins_by_key(R2_RUN / "eval_scores/SV_dev.json", f"chosen_{done['selected']['step']}_SV_dev")}
def nested(scores, name):
    cv.score = macro_score
    try:
        return nested_word_cv(sv, [Candidate(name, frozen(scores), GRID)])
    finally:
        cv.score = positive_score
res = {}
for name, s in sv_scores.items():
    values = np.array(list(s.values()))
    assert np.isfinite(values).all() and np.abs(values).max() < 100, f"{name}: scores outside the cutoff grid"
    res[name] = nested(s, name.replace(" ", "_"))
    assert abs(official_check(gold, res[name]["predictions"]) - res[name]["report"]["average"]) < 1e-12
assert abs(res["round 1"]["report"]["average"] - ROUND1_SV_MACRO) < 1e-9, "Round 1 no longer reproduces 0.7529: STOP and paste this output."
print("Round 1 reproduces its frozen score (0.7529): PASS | official scorer agrees with ours on all three models: PASS\n")
words = sorted(r["word"] for r in res["round 1"]["report"]["per_word"])
per = {name: {r["word"]: r for r in res[name]["report"]["per_word"]} for name in res}
cut = {s["test_group"][1]: s["threshold"] for s in res["round 2"]["selections"]}
better = sum(per["round 2"][w]["f1"] > per["round 1"][w]["f1"] + 1e-9 for w in words)
worse = sum(per["round 2"][w]["f1"] < per["round 1"][w]["f1"] - 1e-9 for w in words)
print(f"{'word':<11}{'Yes rate':>9}{'original':>10}{'round 1':>9}{'round 2':>9}{'change':>9}{'cutoff':>8}   round 2 vs round 1")
for w in words:
    a, b = per["round 1"][w], per["round 2"][w]
    d = b["f1"] - a["f1"]
    print(f"{w:<11}{a['positives'] / a['n']:>9.2f}{per['original'][w]['f1']:>10.3f}{a['f1']:>9.3f}{b['f1']:>9.3f}{d:>+9.3f}{cut[w]:>8.2f}   "
          f"{'better' if d > 1e-9 else ('worse' if d < -1e-9 else 'same')}")
paired = word_bootstrap(res["round 2"]["report"], other=res["round 1"]["report"])
sv_macro = {name: res[name]["report"]["average"] for name in res}
print(f"\nSwedish macro-F1: original {sv_macro['original']:.4f} | round 1 {sv_macro['round 1']:.4f} | round 2 {sv_macro['round 2']:.4f}")
print(f"Round 2 minus round 1: {paired['estimate']:+.4f} (95% range {paired['lower']:+.4f} to {paired['upper']:+.4f}) | "
      f"better on {better}, worse on {worse}, same on {10 - better - worse} of 10 words")
keys = sorted(gold)
def fitted(scores):
    s = np.array([scores[k] for k in keys])
    t = max(GRID, key=lambda x: (macro_score(gold, dict(zip(keys, (s >= x).astype(int).tolist())))["average"], -x))
    return float(t), macro_score(gold, dict(zip(keys, (s >= t).astype(int).tolist())))["average"]
final = {name: fitted(sv_scores[name]) for name in ("round 1", "round 2")}
assert final["round 1"][0] == 0.75, final["round 1"]
print(f"Cutoff fitted on all 10 words: round 1 {final['round 1'][0]:+.2f} (in-sample {final['round 1'][1]:.4f}) | "
      f"round 2 {final['round 2'][0]:+.2f} (in-sample {final['round 2'][1]:.4f})   [in-sample is not the honest estimate]")

# ---------------------------------------------------------------------------
section("3. THE OTHER FIVE LANGUAGES (external test sets; ranking quality = AUC)")
def pooled_macro(y, s, t):
    y, p = np.asarray(y), (np.asarray(s) >= t).astype(int)
    f = lambda c: 2 * np.sum((y == c) & (p == c)) / max(np.sum(y == c) + np.sum(p == c), 1)
    return float((f(1) + f(0)) / 2)
external = {}
for name in ("EN_wic_dev", "RU_axolotl_test_derived"):
    rows = read_jsonl(ENRU_EVAL / f"{name}.jsonl")
    orig = {str(r["id"]): r["yes_minus_no"] for r in json.loads((ENRU_BASE / f"{name}_predictions.json").read_text())}
    r1 = json.loads((R1_RUN / f"{name}_scores.json").read_text())["scores"]
    r2 = json.loads((R2_RUN / f"eval_scores/{name}.json").read_text())["scores"]
    external[name] = ([int(r["label"]) for r in rows], [orig[str(r["id"])] for r in rows],
                      [r1[str(r["id"])]["margin"] for r in rows], [r2[str(r["id"])]["margin"] for r in rows])
for lang in ("IT", "ES", "NL"):
    rows = read_jsonl(XL_EVAL / f"{lang}_xlwsd_test.jsonl")
    orig = json.loads((XL_SCORES / f"{lang}_original_scores.json").read_text())
    r1 = json.loads((XL_SCORES / f"{lang}_trained_scores.json").read_text())
    r2 = json.loads((R2_RUN / f"eval_scores/{lang}_xlwsd_test.json").read_text())["scores"]
    external[f"{lang}_xlwsd_test"] = ([int(r["label"]) for r in rows], [orig[r["id"]] for r in rows],
                                      [r1[r["id"]] for r in rows], [r2[r["id"]]["margin"] for r in rows])
ext_results, auc_ok = {}, True
print(f"{'data':<26}{'AUC original':>13}{'AUC round 1':>13}{'AUC round 2':>13}{'change':>9}{'rule':>7}   macro-F1 round 1 -> round 2 (Swedish cutoffs)")
for name, (y, so, s1, s2) in external.items():
    a0, a1, a2 = roc_auc_score(y, so), roc_auc_score(y, s1), roc_auc_score(y, s2)
    ok = (a2 - a1) >= -0.02
    auc_ok &= ok
    f1, f2 = pooled_macro(y, s1, final["round 1"][0]), pooled_macro(y, s2, final["round 2"][0])
    ext_results[name] = dict(auc_original=float(a0), auc_round1=float(a1), auc_round2=float(a2), auc_change=float(a2 - a1),
                             macro_f1_round1=f1, macro_f1_round2=f2, rule_pass=bool(ok))
    print(f"{name:<26}{a0:>13.4f}{a1:>13.4f}{a2:>13.4f}{a2 - a1:>+9.4f}{'PASS' if ok else 'FAIL':>7}   {f1:.4f} -> {f2:.4f} ({f2 - f1:+.4f})")
print("External sets are not competition data. AUC decides the rule; the macro-F1 columns are for information only.")

# ---------------------------------------------------------------------------
section("4. THE RULE (fixed before training)")
need = ROUND1_SV_MACRO + 0.02
r1a, r1b = sv_macro["round 2"] >= need, better >= 6
r3 = choice["selected"]["step"] == done["selected"]["step"] and "no Swedish dev data" in choice["rule"]
print(f"Rule 1a  Swedish macro-F1 >= {need:.4f} (round 1 + 0.02)     : {sv_macro['round 2']:.4f} -> {'PASS' if r1a else 'FAIL'}")
print(f"Rule 1b  better than round 1 on >= 6 of 10 Swedish words : {better}/10   -> {'PASS' if r1b else 'FAIL'}")
print(f"Rule 2   no language loses more than 0.02 AUC vs round 1 : {'PASS' if auc_ok else 'FAIL'}")
print(f"Rule 3   checkpoint chosen on selection data only        : {'PASS' if r3 else 'FAIL'}")
adopt = bool(r1a and r1b and auc_ok and r3)
save_json(R2_RUN / "adoption_decision.json", dict(
    metric="per-word macro-F1, official scorer commit 0e69238, nested leave-one-word-out",
    swedish=sv_macro, paired_round2_minus_round1=paired, words_better=int(better), words_worse=int(worse),
    per_word={w: dict(round1=per["round 1"][w]["f1"], round2=per["round 2"][w]["f1"]) for w in words},
    fitted_cutoffs={k: dict(cutoff=v[0], in_sample=v[1]) for k, v in final.items()}, external=ext_results,
    rules=dict(r1a=bool(r1a), r1b=bool(r1b), r2=bool(auc_ok), r3=bool(r3)), adopt=adopt,
    note="This file records the decision only. Nothing is frozen or replaced by this cell."))

section("DECISION")
if adopt:
    print(f"ADOPT ROUND 2: Swedish {sv_macro['round 1']:.4f} -> {sv_macro['round 2']:.4f}. It becomes the primary system after a freeze step.")
elif sv_macro["round 2"] > sv_macro["round 1"]:
    print(f"KEEP ROUND 1 AS PRIMARY. Round 2 is higher on Swedish ({sv_macro['round 2']:.4f} vs {sv_macro['round 1']:.4f}) but does not pass the rule.")
    print("It stays available as a second submission in January, because the best submission counts.")
else:
    print(f"KEEP ROUND 1 (Swedish {sv_macro['round 1']:.4f}). Round 2 scored {sv_macro['round 2']:.4f} and is recorded as a tested idea.")
print("Nothing was frozen or changed. Saved:", R2_RUN / "adoption_decision.json")

In [ ]:
"""OPTION B: TWO MORE COPIES OF ROUND 1 (new random seeds). Paste into ONE cell of notebook 12 on a G4 runtime.
Same data and settings as round 1; only the random seed changes (data order, adapter start, dropout).
If Colab disconnects: reconnect to a G4 and run this same cell again. It resumes by itself."""
import sys, gc, json, math, time, uuid, random, shutil, hashlib, subprocess, importlib
from importlib.metadata import version
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
importlib.invalidate_caches()
import numpy as np, torch, torch.nn.functional as F
from peft import LoraConfig, PeftModel, get_peft_model
from peft.import_utils import is_torchao_available
from project_utils import save_json
from semeval_task3.data import load_dataset

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

DEVICE = "cuda"
MODEL_ID = "google/gemma-4-31B-it"
REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"
SEEDS = (2028, 2029)                      # round 1 used 2027
R1_DATA = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
R2_DATA = PROJECT / "outputs/runs/step13_round2_dataset/20261114T214116Z_acd7af91"
FROZEN = PROJECT / "outputs/frozen/primary_gemma4_31b_lora_macro.json"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
ROOT = PROJECT / "outputs/runs/step15_seed_ensemble"
EFFECTIVE_BATCH, LR, SAVE_EVERY, N_TARGET_LAYERS = 16, 1e-4, 100, 230
YES_ID, NO_ID = 10784, 3771

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

# ---------------------------------------------------------------------------
section("1. CHECK EVERY INPUT FILE BEFORE TOUCHING THE MODEL")
is_torchao_available.cache_clear()
assert is_torchao_available() is False, "torchao is still visible to peft: Runtime > Restart session, then run this cell again."
m1, m2 = json.loads((R1_DATA / "manifest.json").read_text()), json.loads((R2_DATA / "manifest.json").read_text())
train_path = R1_DATA / "pilot_train.jsonl"
sel_ru_path = R1_DATA / "pilot_selection_ru_dev.jsonl"
sel_new_path = R2_DATA / "round2_selection_new_languages.jsonl"
assert digest(train_path) == m1["sha256"]["pilot_train.jsonl"], "Round 1 training file changed."
assert digest(sel_ru_path) == m1["sha256"]["pilot_selection_ru_dev.jsonl"], "Russian selection file changed."
assert digest(sel_new_path) == m2["sha256"]["round2_selection_new_languages.jsonl"], "New-language selection file changed."
training = read_rows(train_path)
assert len(training) == 8510
for r in training:
    assert int(r["label"]) in (0, 1) and r["messages"][-1]["role"] == "user"
    assert all(m["role"] in ("system", "user") for m in r["messages"])
selection = {"selection_ru": read_rows(sel_ru_path), "selection_new_languages": read_rows(sel_new_path)}
frozen_cfg = json.loads(FROZEN.read_text())
R1_ADAPTER = Path(frozen_cfg["adapter_dir"])
assert digest(R1_ADAPTER / "adapter_model.safetensors") == frozen_cfg["adapter_sha256"], "Round 1 adapter changed."
enru_manifest = json.loads((ENRU_EVAL / "manifest.json").read_text())
xl_manifest = json.loads((XL_EVAL / "manifest.json").read_text())
eval_sets = {}
for name in ("EN_wic_dev", "RU_axolotl_test_derived"):
    assert digest(ENRU_EVAL / f"{name}.jsonl") == enru_manifest["output_sha256"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(ENRU_EVAL / f"{name}.jsonl")
for lang in ("IT", "ES", "NL"):
    name = f"{lang}_xlwsd_test"
    assert digest(XL_EVAL / f"{name}.jsonl") == xl_manifest["files"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(XL_EVAL / f"{name}.jsonl")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=False)
eval_sets["SV_dev"] = [dict(id="|".join(u.key), messages=[
    dict(role="system", content="You are an expert lexicographer of historical Swedish."),
    dict(role="user", content=(f"Target word (lemma): {u.word}\n"
                               f"Sense definition (Swedish): {u.definition}\n"
                               f"Sentence from {u.year}: {u.marked_text()}\n\n"
                               "Does the word marked with <t> and </t> in the sentence express the sense given by the definition? "
                               "Answer only Yes or No."))]) for u in sv.usages]
assert len(eval_sets["SV_dev"]) == 1983
print(f"Round 1 training file : hash PASS | {len(training):,} examples (the same file round 1 trained on)")
print(f"Round 1 adapter       : hash PASS ({frozen_cfg['adapter_sha256'][:16]}...)")
print(f"Selection sets        : hash PASS | { {k: len(v) for k, v in selection.items()} } (scored only, never used to train)")
print(f"Evaluation sets       : hash PASS | { {k: len(v) for k, v in eval_sets.items()} }")
TOTAL = math.ceil(len(training) / EFFECTIVE_BATCH)
WARMUP = math.ceil(0.05 * TOTAL)
print(f"Each seed: {TOTAL} updates (one pass), warm-up {WARMUP}, then the final adapter is scored. Seeds: {SEEDS}")

# ---------------------------------------------------------------------------
section("2. GEMMA 4 31B (bf16)")
if "gemma_model" in globals() or "full_model" in globals() or "BASE" in globals():
    print("Gemma is already loaded in this session: reusing it.")
else:
    from transformers import AutoProcessor, AutoModelForMultimodalLM
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 2**30:.0f} GiB | transformers {version('transformers')}")
    assert props.total_memory / 2**30 > 70, "Gemma 31B training needs a GPU with more than 70 GB: use the G4."
    print("Loading Gemma (about 5 minutes on the G4) ...", flush=True)
    gemma_processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
    gemma_tokenizer = gemma_processor.tokenizer
    gemma_model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, revision=REVISION, dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
    print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
gemma_tokenizer.padding_side = "left"
assert gemma_tokenizer.encode("Yes", add_special_tokens=False) == [YES_ID]
assert gemma_tokenizer.encode("No", add_special_tokens=False) == [NO_ID]

def plain_base():
    """The original Gemma with every adapter removed (unload() removes adapters without merging them)."""
    current = None
    for name in ("full_model", "BASE", "gemma_model"):
        if globals().get(name) is not None:
            current = globals()[name]
            break
    base = current.unload() if isinstance(current, PeftModel) else current
    for name in ("full_model", "gemma_model", "model", "full_optimizer", "params"):
        globals().pop(name, None)
    assert base.config._name_or_path == MODEL_ID and not any("lora_" in n for n, _ in base.named_parameters())
    gc.collect()
    torch.cuda.empty_cache()
    return base

def encode(batch):
    inputs = gemma_processor.apply_chat_template(
        [r["messages"] for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
        processor_kwargs={"padding": True, "truncation": False}).to(DEVICE)
    assert inputs["attention_mask"][:, -1].all(), "Padding is not on the left."
    return inputs

def score_rows(model, rows, destination, tag):
    """Resumable scoring: labels never enter the model; saved to Drive every 100 batches."""
    identity = hashlib.sha256(json.dumps([(str(r["id"]), r["messages"]) for r in rows],
                                         ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    if destination.exists():
        saved = json.loads(destination.read_text())
        assert saved["input_sha256"] == identity and saved["model_tag"] == tag, f"{destination.name}: saved scores belong to other inputs."
    else:
        saved = dict(input_sha256=identity, model_tag=tag, scores={})
    pending = [r for r in rows if str(r["id"]) not in saved["scores"]]
    model.eval()
    t0, n_batches = time.perf_counter(), math.ceil(len(pending) / 2)
    try:
        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.bfloat16):
            for b, i in enumerate(range(0, len(pending), 2), 1):
                batch = pending[i:i + 2]
                logits = model(**encode(batch), use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
                assert torch.isfinite(logits).all()
                for r, margin in zip(batch, (logits[:, YES_ID] - logits[:, NO_ID]).tolist()):
                    saved["scores"][str(r["id"])] = dict(margin=float(margin))
                del logits
                if b % 100 == 0:
                    save_json(destination, saved)
    finally:
        save_json(destination, saved)
    assert set(saved["scores"]) == {str(r["id"]) for r in rows}
    if pending:
        print(f"      {tag}: {len(rows):,} scores ({(time.perf_counter() - t0) / 60:.1f} min)", flush=True)
    return saved

# ---------------------------------------------------------------------------
section("3. ROUND 1 ADAPTER ON THE NEW-LANGUAGE SELECTION SET (for the cutoff work later; about 1.5 minutes)")
r1_dir = ROOT / "seed_2027_round1"
r1_dir.mkdir(parents=True, exist_ok=True)
r1_file = r1_dir / "selection_new_languages_scores.json"
if r1_file.exists() and len(json.loads(r1_file.read_text())["scores"]) == len(selection["selection_new_languages"]):
    print("Already scored: skipping.")
else:
    BASE = plain_base()
    full_model = PeftModel.from_pretrained(BASE, str(R1_ADAPTER), is_trainable=False)
    score_rows(full_model, selection["selection_new_languages"], r1_file, "round1_seed_2027_selection_new_languages")
    BASE = plain_base()
print("Round 1 selection scores saved.")

# ---------------------------------------------------------------------------
def train_seed(seed):
    run = ROOT / f"seed_{seed}"
    run.mkdir(parents=True, exist_ok=True)
    config = dict(model=MODEL_ID, revision=REVISION, seed=seed, recipe="round 1 (step09) unchanged except the seed",
                  examples=len(training), train_sha256=digest(train_path), micro_batch=1, effective_batch=EFFECTIVE_BATCH,
                  learning_rate=LR, warmup_steps=WARMUP, steps=TOTAL, schedule="linear warmup then linear decay",
                  r=16, alpha=32, dropout=0.05, target="language-model q/k/v/o", base_precision="BF16, no quantization",
                  checkpoint="final update (fixed before training; round 1's selection also chose its final update)")
    if (run / "config.json").exists():
        assert json.loads((run / "config.json").read_text()) == json.loads(json.dumps(config)), \
            f"seed {seed}: settings changed since this run started. Paste this message."
    else:
        save_json(run / "config.json", config)
    if (run / "COMPLETE.json").exists():
        print(f"Seed {seed}: already complete, skipping.")
        return
    global BASE, full_model
    BASE = plain_base()
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    order = list(training)
    random.Random(seed).shuffle(order)
    targets = [n for n, m in BASE.named_modules() if isinstance(m, torch.nn.Linear) and "language_model" in n.split(".")
               and n.split(".")[-1] in ("q_proj", "k_proj", "v_proj", "o_proj")]
    assert len(targets) == N_TARGET_LAYERS, f"Unexpected number of attention layers: {len(targets)}"
    latest_file = run / "latest.json"
    final_dir = run / "adapter_final"
    state = None
    if (final_dir / "adapter_config.json").exists():
        full_model = PeftModel.from_pretrained(BASE, str(final_dir), is_trainable=False)
        step_done, log = TOTAL, json.loads((run / "training_log.json").read_text())
        print(f"Seed {seed}: training already finished; continuing with scoring.")
    elif latest_file.exists():
        latest = Path(json.loads(latest_file.read_text())["directory"])
        full_model = PeftModel.from_pretrained(BASE, str(latest / "adapter"), is_trainable=True)
        state = torch.load(latest / "training_state.pt", map_location="cpu", weights_only=False)     # our own file
        step_done, log = state["step"], state["log"]
        print(f"Seed {seed}: resuming after update {step_done} of {TOTAL}.")
    else:
        full_model = get_peft_model(BASE, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=targets, bias="none"))
        step_done, log = 0, []
        assert all(torch.count_nonzero(p).item() == 0 for n, p in full_model.named_parameters() if "lora_B" in n)
        print(f"Seed {seed}: fresh adapter attached (starts from zero).")

    if step_done < TOTAL:
        full_model.config.use_cache = False
        full_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        params = [p for p in full_model.parameters() if p.requires_grad]
        assert params and all("lora_" in n for n, p in full_model.named_parameters() if p.requires_grad)
        optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
        if state:
            optimizer.load_state_dict(state["optimizer"])
            random.setstate(state["python_rng"])
            np.random.set_state(state["numpy_rng"])
            torch.set_rng_state(state["torch_rng"])
            if state.get("cuda_rng") is not None:
                torch.cuda.set_rng_state_all(state["cuda_rng"])
        del state

        def save_resume(step):
            folder = run / f"resume_{step:04d}_{uuid.uuid4().hex[:8]}"
            folder.mkdir()
            full_model.save_pretrained(folder / "adapter", safe_serialization=True)
            torch.save(dict(step=step, log=log, optimizer=optimizer.state_dict(), python_rng=random.getstate(),
                            numpy_rng=np.random.get_state(), torch_rng=torch.get_rng_state(),
                            cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None),
                       folder / "training_state.pt")
            previous = json.loads(latest_file.read_text())["directory"] if latest_file.exists() else None
            save_json(latest_file, {"directory": str(folder), "step": step})
            if previous and Path(previous).parent == run and Path(previous).name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
            save_json(run / "training_log.json", log)

        full_model.train()
        t0 = time.perf_counter()
        for step in range(step_done + 1, TOTAL + 1):
            group = order[(step - 1) * EFFECTIVE_BATCH:step * EFFECTIVE_BATCH]
            scale = step / WARMUP if step <= WARMUP else (TOTAL - step + 1) / (TOTAL - WARMUP)
            for pg in optimizer.param_groups:
                pg["lr"] = LR * scale
            optimizer.zero_grad(set_to_none=True)
            loss_sum, correct = 0.0, 0
            for r in group:
                answer = torch.tensor([YES_ID if int(r["label"]) else NO_ID], device=DEVICE)
                with torch.autocast(DEVICE, dtype=torch.bfloat16):
                    logits = full_model(**encode([r]), use_cache=False, logits_to_keep=1).logits[:, -1, :]
                loss = F.cross_entropy(logits.float(), answer)
                assert torch.isfinite(loss), "Non-finite loss: stop and paste this message. The last saved checkpoint is intact."
                (loss / len(group)).backward()
                loss_sum += float(loss.detach())
                correct += int((logits[0, YES_ID] > logits[0, NO_ID]).item() == bool(int(r["label"])))
                del logits, loss, answer
            norm = float(torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True))
            optimizer.step()
            log.append(dict(step=step, loss=loss_sum / len(group), batch_accuracy=correct / len(group), grad_norm=norm, lr=LR * scale))
            if step == 1 or step % 50 == 0 or step == TOTAL:
                minutes = (time.perf_counter() - t0) / 60
                print(f"   seed {seed} | update {step:>3}/{TOTAL}: loss {loss_sum / len(group):.4f} | accuracy {correct / len(group):.2f} | "
                      f"about {minutes / (step - step_done) * (TOTAL - step):.0f} min of training left", flush=True)
            if step % SAVE_EVERY == 0 and step < TOTAL:
                save_resume(step)
        full_model.save_pretrained(final_dir, safe_serialization=True)
        save_json(run / "training_log.json", log)
        optimizer.zero_grad(set_to_none=True)
        del optimizer, params
        if latest_file.exists():
            previous = Path(json.loads(latest_file.read_text())["directory"])
            if previous.parent == run and previous.name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
            latest_file.unlink()
        gc.collect()
        torch.cuda.empty_cache()
        losses = [x["loss"] for x in log]
        print(f"Seed {seed}: training done | mean loss first 50 updates {np.mean(losses[:50]):.3f} -> last 50 updates {np.mean(losses[-50:]):.3f}")

    print(f"Seed {seed}: scoring the final adapter on 6 evaluation sets and 2 selection sets (about 14 minutes)", flush=True)
    (run / "eval_scores").mkdir(exist_ok=True)
    for name, rows in list(eval_sets.items()) + list(selection.items()):
        score_rows(full_model, rows, run / "eval_scores" / f"{name}.json", f"seed_{seed}_{name}")
    save_json(run / "COMPLETE.json", dict(seed=seed, adapter=str(final_dir), adapter_sha256=digest(final_dir / "adapter_model.safetensors"),
                                          eval_scores={name: f"eval_scores/{name}.json" for name in list(eval_sets) + list(selection)}))
    print(f"Seed {seed}: COMPLETE")
    BASE = plain_base()

for i, seed in enumerate(SEEDS, 4):
    section(f"{i}. SEED {seed}: TRAIN ROUND 1 AGAIN, THEN SCORE (about 1 hour)")
    train_seed(seed)

section("SUMMARY")
for seed in SEEDS:
    print(f"Seed {seed}: {'COMPLETE' if (ROOT / f'seed_{seed}' / 'COMPLETE.json').exists() else 'NOT complete'}")
print(f"Everything is saved in: {ROOT}")
print("The current primary system (round 1, Swedish 0.753) is untouched.")
print("Next: the decision cell (it averages the three models and applies the rule).")

In [ ]:
"""OPTION B DECISION. Run AFTER the seed cell has finished (no GPU needed, about 2 minutes).
Averages the three round 1 copies (seeds 2027, 2028, 2029) and applies the rule fixed before training."""
import sys, json, subprocess, importlib.util
from collections import defaultdict
from pathlib import Path
import numpy as np
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
from sklearn.metrics import roc_auc_score
from project_utils import save_json, frozen
from semeval_task3.data import load_dataset
import semeval_task3.cv as cv
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score as positive_score, word_bootstrap

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

R1_RUN = PROJECT / "outputs/runs/step09_lora_full_pilot/20261105T232211Z_6b04a51b"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
XL_SCORES = PROJECT / "outputs/runs/step11_xlwsd_scores/adapter_6f3d236ac50e"
ROOT = PROJECT / "outputs/runs/step15_seed_ensemble"
SEEDS = (2028, 2029)
SCORER_DIR, SCORER_COMMIT = Path("/content/official-scorer-latest"), "0e69238"
ROUND1_SV_MACRO = 0.7529025040663228
GRID = tuple([-1000.0] + np.round(np.arange(-100, 100.001, 0.25), 2).tolist() + [1000.0])
EXTERNAL = ("EN_wic_dev", "RU_axolotl_test_derived", "IT_xlwsd_test", "ES_xlwsd_test", "NL_xlwsd_test")

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

# ---------------------------------------------------------------------------
section("1. LOAD THE SCORES OF ALL THREE MODELS")
for seed in SEEDS:
    assert (ROOT / f"seed_{seed}" / "COMPLETE.json").exists(), f"Seed {seed} is not finished. Run this cell after the seed cell's SUMMARY."
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = sv.require_gold()
def by_key(saved, tag=None):
    if tag:
        assert saved["model_tag"] == tag, f"Unexpected scores: {saved['model_tag']}"
    return {u.key: float(saved["scores"]["|".join(u.key)]["margin"]) for u in sv.usages}
sv_scores = {"seed 2027 (round 1)": by_key(json.loads((R1_RUN / "SV_scores.json").read_text()), "chosen_532_SV")}
for seed in SEEDS:
    sv_scores[f"seed {seed}"] = by_key(json.loads((ROOT / f"seed_{seed}/eval_scores/SV_dev.json").read_text()), f"seed_{seed}_SV_dev")
SINGLES = list(sv_scores)
sv_scores["ensemble (average of 3)"] = {k: float(np.mean([sv_scores[s][k] for s in SINGLES])) for k in gold}
for name, s in sv_scores.items():
    v = np.array(list(s.values()))
    assert np.isfinite(v).all() and np.abs(v).max() < 100, f"{name}: scores outside the cutoff grid"
r = np.corrcoef([[sv_scores[s][k] for k in sorted(gold)] for s in SINGLES])
print(f"Swedish scores loaded for {len(SINGLES)} models + their average")
print(f"How similar the three models are (correlation of Swedish scores): "
      f"{r[0, 1]:.3f}, {r[0, 2]:.3f}, {r[1, 2]:.3f}   (lower = more room for averaging to help)")

def macro_score(gold, predictions):
    """Official metric: per word, the mean of Yes-F1 and No-F1; then the mean over words."""
    report = positive_score(gold, predictions)
    languages = defaultdict(list)
    for row in report["per_word"]:
        tp, fp, fn = row["tp"], row["fp"], row["fn"]
        tn = row["n"] - tp - fp - fn
        parts = []
        if tp + fp + fn > 0:
            parts.append(2 * tp / (2 * tp + fp + fn))
        if tn + fp + fn > 0:
            parts.append(2 * tn / (2 * tn + fn + fp))
        row["positive_f1"], row["f1"] = row["f1"], sum(parts) / len(parts)
        languages[row["language"]].append(row["f1"])
    report["per_language"] = {k: sum(v) / len(v) for k, v in languages.items()}
    report["average"] = sum(report["per_language"].values()) / len(report["per_language"])
    return report

if not SCORER_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/ChangeIsKey/semeval2027-task3.git", str(SCORER_DIR)], check=True)
subprocess.run(["git", "-C", str(SCORER_DIR), "checkout", "-q", SCORER_COMMIT], check=True)
assert "average='macro'" in (SCORER_DIR / "evaluation.py").read_text()
sys.path.insert(0, str(SCORER_DIR))
spec = importlib.util.spec_from_file_location("official_eval_latest", SCORER_DIR / "evaluation.py")
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
def official_check(predictions):
    return official.compute_f1_score({f"{k[1]};{k[2]}": int(v) for k, v in gold.items()},
                                     {f"{k[1]};{k[2]}": int(predictions[k]) for k in gold}, "check")

# ---------------------------------------------------------------------------
section("2. SWEDISH DEV, OFFICIAL METRIC (nested leave-one-word-out, same procedure as before)")
def nested(scores, name):
    cv.score = macro_score
    try:
        return nested_word_cv(sv, [Candidate(name, frozen(scores), GRID)])
    finally:
        cv.score = positive_score
res = {}
for name, s in sv_scores.items():
    res[name] = nested(s, name.split()[0] + "_" + name.split()[1].strip("()"))
    assert abs(official_check(res[name]["predictions"]) - res[name]["report"]["average"]) < 1e-12
R1, ENS = SINGLES[0], "ensemble (average of 3)"
assert abs(res[R1]["report"]["average"] - ROUND1_SV_MACRO) < 1e-9, "Round 1 no longer reproduces 0.7529: STOP and paste this output."
print("Round 1 reproduces 0.7529: PASS | official scorer agrees with ours on all four: PASS\n")
singles = [res[s]["report"]["average"] for s in SINGLES]
for name in res:
    print(f"   {name:<26} macro-F1 {res[name]['report']['average']:.4f}")
print(f"\nRun-to-run spread of a single model: {min(singles):.4f} to {max(singles):.4f} (mean {np.mean(singles):.4f})")
print("This shows how much of any score is luck of the random seed.")
words = sorted(row["word"] for row in res[R1]["report"]["per_word"])
per = {name: {row["word"]: row["f1"] for row in res[name]["report"]["per_word"]} for name in res}
better = sum(per[ENS][w] > per[R1][w] + 1e-9 for w in words)
worse = sum(per[ENS][w] < per[R1][w] - 1e-9 for w in words)
print(f"\n{'word':<11}" + "".join(f"{n.split(' (')[0]:>11}" for n in SINGLES) + f"{'average':>10}{'change vs round 1':>19}")
for w in words:
    print(f"{w:<11}" + "".join(f"{per[n][w]:>11.3f}" for n in SINGLES) + f"{per[ENS][w]:>10.3f}{per[ENS][w] - per[R1][w]:>+19.3f}")
paired = word_bootstrap(res[ENS]["report"], other=res[R1]["report"])
print(f"\nAverage of 3 minus round 1: {paired['estimate']:+.4f} (95% range {paired['lower']:+.4f} to {paired['upper']:+.4f}) | "
      f"better on {better}, worse on {worse}, same on {10 - better - worse} of 10 words")
keys = sorted(gold)
def fitted(scores):
    s = np.array([scores[k] for k in keys])
    t = max(GRID, key=lambda x: (macro_score(gold, dict(zip(keys, (s >= x).astype(int).tolist())))["average"], -x))
    return float(t), macro_score(gold, dict(zip(keys, (s >= t).astype(int).tolist())))["average"]
cutoff = {name: fitted(sv_scores[name]) for name in (R1, ENS)}
assert cutoff[R1][0] == 0.75, cutoff[R1]
print(f"Cutoff fitted on all 10 words: round 1 {cutoff[R1][0]:+.2f} | average of 3 {cutoff[ENS][0]:+.2f} (in-sample {cutoff[ENS][1]:.4f})")

# ---------------------------------------------------------------------------
section("3. THE OTHER FIVE LANGUAGES: is any language worse? (AUC and macro-F1, both must not drop > 0.02)")
def pooled_macro(y, s, t):
    y, p = np.asarray(y), (np.asarray(s) >= t).astype(int)
    f = lambda c: 2 * np.sum((y == c) & (p == c)) / max(np.sum(y == c) + np.sum(p == c), 1)
    return float((f(1) + f(0)) / 2)
ext, all_ok = {}, True
print(f"{'data':<26}{'AUC r1':>8}{'AUC avg':>9}{'change':>9}   {'mF1 r1':>8}{'mF1 avg':>9}{'change':>9}   rule   (single seeds' AUC)")
for name in EXTERNAL:
    rows = read_jsonl((ENRU_EVAL if name[:2] in ("EN", "RU") else XL_EVAL) / f"{name}.jsonl")
    ids, y = [str(row["id"]) for row in rows], [int(row["label"]) for row in rows]
    if name[:2] in ("EN", "RU"):
        r1 = json.loads((R1_RUN / f"{name}_scores.json").read_text())["scores"]
        r1 = [r1[i]["margin"] for i in ids]
    else:
        r1 = json.loads((XL_SCORES / f"{name[:2]}_trained_scores.json").read_text())
        r1 = [r1[i] for i in ids]
    seeds = [r1] + [[json.loads((ROOT / f"seed_{s}/eval_scores/{name}.json").read_text())["scores"][i]["margin"] for i in ids] for s in SEEDS]
    avg = list(np.mean(seeds, axis=0))
    a1, a2 = roc_auc_score(y, r1), roc_auc_score(y, avg)
    f1, f2 = pooled_macro(y, r1, cutoff[R1][0]), pooled_macro(y, avg, cutoff[ENS][0])
    ok = (a2 - a1) >= -0.02 and (f2 - f1) >= -0.02
    all_ok &= ok
    ext[name] = dict(auc_round1=a1, auc_average=a2, macro_f1_round1=f1, macro_f1_average=f2, rule_pass=bool(ok),
                     auc_single_seeds=[float(roc_auc_score(y, s)) for s in seeds])
    print(f"{name:<26}{a1:>8.4f}{a2:>9.4f}{a2 - a1:>+9.4f}   {f1:>8.4f}{f2:>9.4f}{f2 - f1:>+9.4f}   {'PASS' if ok else 'FAIL'}   "
          + " ".join(f"{x:.3f}" for x in ext[name]["auc_single_seeds"]))
print("macro-F1 uses each system's own Swedish cutoff. External sets are not competition data.")

# ---------------------------------------------------------------------------
section("4. THE RULE (fixed before training)")
avg_sv = res[ENS]["report"]["average"]
r1a, r1b = avg_sv >= ROUND1_SV_MACRO + 0.02, better >= 6
print(f"Rule 1a  Swedish macro-F1 >= {ROUND1_SV_MACRO + 0.02:.4f} (round 1 + 0.02)      : {avg_sv:.4f} -> {'PASS' if r1a else 'FAIL'}")
print(f"Rule 1b  better than round 1 on >= 6 of 10 Swedish words  : {better}/10   -> {'PASS' if r1b else 'FAIL'}")
print(f"Rule 2   no other language loses > 0.02 AUC or macro-F1    : {'PASS' if all_ok else 'FAIL'}")
adopt = bool(r1a and r1b and all_ok)
save_json(ROOT / "ensemble_decision.json", dict(
    metric="per-word macro-F1, official scorer commit 0e69238, nested leave-one-word-out",
    swedish={name: res[name]["report"]["average"] for name in res}, single_seed_spread=[min(singles), max(singles)],
    paired_average_minus_round1=paired, words_better=int(better), words_worse=int(worse),
    per_word=per, fitted_cutoffs={k: dict(cutoff=v[0], in_sample=v[1]) for k, v in cutoff.items()},
    external=ext, rules=dict(r1a=bool(r1a), r1b=bool(r1b), r2=bool(all_ok)), adopt=adopt,
    note="Records the decision only. Nothing is frozen or replaced by this cell."))

section("DECISION")
if adopt:
    print(f"ADOPT THE AVERAGE OF 3 MODELS: Swedish {ROUND1_SV_MACRO:.4f} -> {avg_sv:.4f}, no language worse. It becomes primary after a freeze step.")
elif avg_sv > ROUND1_SV_MACRO and all_ok:
    print(f"KEEP ROUND 1 AS PRIMARY. The average is higher ({avg_sv:.4f} vs {ROUND1_SV_MACRO:.4f}) with no language worse, "
          "but below the +0.02 bar. It is a strong second submission for January (the best submission counts).")
else:
    print(f"KEEP ROUND 1 (Swedish {ROUND1_SV_MACRO:.4f}). The average scored {avg_sv:.4f}.")
print("Nothing was frozen or changed. Saved:", ROOT / "ensemble_decision.json")

In [ ]:
import sys, json, hashlib
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
if str(PROJECT / "notebooks") not in sys.path:
    sys.path.insert(0, str(PROJECT / "notebooks"))
from project_utils import save_json

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

PRIMARY = PROJECT / "outputs/frozen/primary_gemma4_31b_lora_macro.json"
ROOT = PROJECT / "outputs/runs/step15_seed_ensemble"
NEW = PROJECT / "outputs/frozen/system2_ensemble3_gemma4_31b_lora.json"
R1, ENS = "seed 2027 (round 1)", "ensemble (average of 3)"
digest = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()

section("1. CHECK THE THREE ADAPTERS (fingerprints read from the files themselves)")
primary = json.loads(PRIMARY.read_text())
decision = json.loads((ROOT / "ensemble_decision.json").read_text())
adapters = [dict(seed=2027, adapter_dir=primary["adapter_dir"], sha256=primary["adapter_sha256"], note="round 1 = system 1")]
for seed in (2028, 2029):
    done = json.loads((ROOT / f"seed_{seed}/COMPLETE.json").read_text())
    adapters.append(dict(seed=seed, adapter_dir=done["adapter"], sha256=done["adapter_sha256"]))
for a in adapters:
    actual = digest(Path(a["adapter_dir"]) / "adapter_model.safetensors")
    assert actual == a["sha256"], f"Seed {a['seed']}: adapter file changed. STOP and paste this output."
    print(f"Seed {a['seed']}: fingerprint PASS ({actual[:16]}...)")
assert len({a["sha256"] for a in adapters}) == 3, "Two adapters are identical: STOP."

section("2. WRITE THE SYSTEM 2 CONFIG (system 1 stays the primary system)")
cutoff = decision["fitted_cutoffs"][ENS]["cutoff"]
assert cutoff == 0.75, f"Unexpected cutoff {cutoff}: STOP and paste this output."
assert not decision["adopt"], "The decision file says ADOPT: this freeze cell expects the 'keep round 1' outcome. STOP."
assert not NEW.exists(), "System 2 config already exists: STOP, do not overwrite."
config = dict(
    status="SYSTEM 2 (second submission). System 1 (round 1 alone) stays primary: the pre-registered rule was not met.",
    model_id=primary["model_id"], revision=primary["revision"],
    adapters=adapters,
    combine="score = mean over the 3 adapters of logit(Yes) - logit(No); predict 1 if score >= final_threshold",
    final_threshold=cutoff,
    threshold_rule="single cutoff fitted on all 10 Swedish dev words for macro-F1, predeclared grid, ties -> lower cutoff",
    prompt=primary["prompt"], scoring=primary["scoring"],
    training="each adapter: round 1 recipe and data (8,510 RU+EN examples), final update; only the random seed differs",
    metric="per-word macro-F1 (official scorer commit 0e69238)",
    swedish_nested_cv_macro_f1=decision["swedish"][ENS],
    swedish_in_sample_macro_f1=decision["fitted_cutoffs"][ENS]["in_sample"],
    swedish_single_models=dict((k, v) for k, v in decision["swedish"].items() if k != ENS),
    paired_change_vs_system1=decision["paired_average_minus_round1"],
    external_checks=decision["external"],
    decision_file=str(ROOT / "ensemble_decision.json"), system1_config=str(PRIMARY.relative_to(PROJECT)))
save_json(NEW, config)
print(f"Cutoff                      : {cutoff}")
print(f"Swedish macro-F1 (nested)   : system 2 {config['swedish_nested_cv_macro_f1']:.4f} | system 1 {decision['swedish'][R1]:.4f}")
print(f"Single models               : " + ", ".join(f"{k} {v:.4f}" for k, v in config["swedish_single_models"].items()))
print(f"Saved                       : {NEW}")

section("SUMMARY")
print("System 1 (primary)          : round 1 alone, cutoff 0.75  ->", PRIMARY.name)
print("System 2 (second submission): average of 3 adapters, cutoff 0.75 ->", NEW.name)
print("Nothing else was changed. You can now disconnect the runtime.")